# Setup


In [3]:
import json
import math
import re
from itertools import combinations
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, ScalarFormatter
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
from statsmodels.stats.oneway import anova_oneway


In [4]:
import matplotlib as mpl

mpl.rcParams.update({
    "font.size": 8,          # base font
    "axes.labelsize": 8,
    "axes.titlesize": 9,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
})

TEXTWIDTH_IN = 5.8          # approximate \textwidth
QQ_GRID_HEIGHT_IN = 8.0

# Constants


In [5]:
DATA_DIR = Path("/Users/casperkristiansson/Thesis Data/logs_4")

DIAGNOSTICS_DIR = Path("figures")
QQ_DIR = DIAGNOSTICS_DIR / "qq"
DRIFT_DIR = DIAGNOSTICS_DIR / "drift"
SUMMARY_DIR = DIAGNOSTICS_DIR

ARTIFACTS_DIR = Path("")
TABLE_DIR = ARTIFACTS_DIR / "tables"
FIGURE_DIR = ARTIFACTS_DIR / "figures"

for path in (QQ_DIR, DRIFT_DIR, SUMMARY_DIR, TABLE_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

FORMATS = ["hdf5", "zarr", "tiledb", "root"]
CODECS = ["gzip", "lz4", "zstd"]
PATTERNS = ["slice", "full"]
ROWS = [(codec, pattern) for codec in CODECS for pattern in PATTERNS]

RNG_SEED = 979969114
BOOTSTRAP_DRAWS = 10_000

RUN_RX = re.compile(r"^(?P<fmt>hdf5|zarr|tiledb|root)_(?P<codec>gzip|lz4|zstd)_(?P<pattern>slice|full)_(?P<runid>\d{8}-\d{6})\.json$")
CW_RX = re.compile(r"^(?P<fmt>hdf5|zarr|tiledb|root)_(?P<codec>gzip|lz4|zstd)_(?P<pattern>slice|full)_cloudwatch\.json$")


# Helper Functions


In [8]:
def parse_run_meta(name: str) -> dict[str, str] | None:
    match = RUN_RX.match(name)
    return match.groupdict() if match else None


def parse_cloudwatch_meta(name: str) -> dict[str, str] | None:
    match = CW_RX.match(name)
    return match.groupdict() if match else None


def load_runs(path: Path) -> pd.DataFrame:
    with open(path, "r") as handle:
        payload = json.load(handle)
    frame = pd.DataFrame(payload["data"])
    if "i" not in frame.columns:
        frame["i"] = np.arange(1, len(frame) + 1)
    frame["time_sec"] = frame["t_total_ns"] / 1e9
    frame["ln_time"] = np.log(frame["time_sec"])
    return frame


def load_latency_runs(data_dir: Path, codec: str, pattern: str) -> pd.DataFrame:
    rows = []
    for path in data_dir.iterdir():
        if not (path.is_file() and path.suffix == ".json" and "_cloudwatch" not in path.name):
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        if meta["codec"] != codec or meta["pattern"] != pattern:
            continue
        frame = load_runs(path).loc[:, ["i", "time_sec", "ln_time"]].copy()
        frame["format"] = meta["fmt"]
        frame["run_id"] = meta["runid"]
        rows.append(frame)
    runs = pd.concat(rows, ignore_index=True)
    runs = runs[runs["format"].isin(FORMATS)]
    return runs


def qq_envelope(n: int, draws: int = 2000, seed: int = RNG_SEED) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    samples = np.sort(rng.standard_normal(size=(draws, n)), axis=1)
    lower = np.percentile(samples, 2.5, axis=0)
    upper = np.percentile(samples, 97.5, axis=0)
    theoretical = stats.norm.ppf((np.arange(1, n + 1) - 0.5) / n)
    return theoretical, lower, upper


def lag1_autocorr(values: np.ndarray) -> float:
    centered = values - values.mean()
    denom = np.dot(centered, centered)
    if denom == 0:
        return 0.0
    return float(np.dot(centered[:-1], centered[1:]) / denom)


def smooth_series(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    if len(x) == 0:
        return np.array([])
    try:
        return sm_lowess(y, x, frac=0.3, return_sorted=False)
    except Exception:
        window = max(3, int(round(0.3 * len(x))))
        if window % 2 == 0:
            window += 1
        pad = window // 2
        padded = np.pad(y, (pad, pad), mode="edge")
        kernel = np.ones(window) / window
        return np.convolve(padded, kernel, mode="valid")


def nice_ylim(ymin: float, ymax: float, pad_frac: float = 0.1, floor: float | None = None) -> tuple[float, float]:
    if not np.isfinite(ymin) or not np.isfinite(ymax):
        base = floor if floor is not None else 0.0
        return base, base + 1.0
    if ymax <= ymin:
        ymax = ymin + 1.0
    span = ymax - ymin
    pad = pad_frac * span if span > 0 else 1.0
    low = ymin - pad
    high = ymax + pad
    if floor is not None:
        low = max(floor, low)
    return low, high


def garwood_rate_ci(k: float, exposure: float, alpha: float = 0.05) -> tuple[float, float]:
    if not np.isfinite(exposure) or exposure <= 0 or not np.isfinite(k) or k < 0:
        return np.nan, np.nan
    lower = 0.0 if k == 0 else 0.5 * stats.chi2.ppf(alpha / 2, 2 * k)
    upper = 0.5 * stats.chi2.ppf(1 - alpha / 2, 2 * (k + 1))
    return lower / exposure, upper / exposure


def gmean_ci_from_ln(values: np.ndarray, alpha: float = 0.05) -> tuple[float, float, float]:
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    mean = float(values.mean())
    std = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    if len(values) > 1 and np.isfinite(std) and std > 0:
        tcrit = stats.t.ppf(1 - alpha / 2, len(values) - 1)
        delta = tcrit * std / math.sqrt(len(values))
        return math.exp(mean), math.exp(mean - delta), math.exp(mean + delta)
    val = math.exp(mean)
    return val, val, val


def bootstrap_quantile(values: np.ndarray, q: float, draws: int, seed: int) -> tuple[float, float, float]:
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(draws, len(values)), replace=True)
    estimates = np.quantile(samples, q, axis=1)
    point = float(np.quantile(values, q))
    lo, hi = np.quantile(estimates, [0.025, 0.975])
    return point, float(lo), float(hi)


def welch_anova_ln(groups: dict[str, np.ndarray]) -> dict[str, float]:
    arrays = [np.asarray(v, dtype=float) for v in groups.values() if len(v) > 0]
    if len(arrays) < 2:
        return {"F": np.nan, "df_between": np.nan, "df_within": np.nan, "pvalue": np.nan}
    res = anova_oneway(arrays, use_var="unequal", welch_correction=True)
    return {
        "F": float(res.statistic),
        "df_between": float(res.df_num),
        "df_within": float(res.df_denom),
        "pvalue": float(res.pvalue),
    }


def omega_sq_oneway_ln(groups: dict[str, np.ndarray]) -> float:
    arrays = [np.asarray(v, dtype=float) for v in groups.values() if len(v) > 0]
    if len(arrays) < 2:
        return float("nan")
    means = [float(arr.mean()) for arr in arrays]
    ns = [len(arr) for arr in arrays]
    all_values = np.concatenate(arrays)
    grand = float(all_values.mean())
    ss_within = sum(np.sum((arr - mean) ** 2) for arr, mean in zip(arrays, means))
    ss_between = sum(n * (mean - grand) ** 2 for n, mean in zip(ns, means))
    sst = ss_within + ss_between
    k = len(arrays)
    total_n = sum(ns)
    if total_n <= k:
        return float("nan")
    ms_within = ss_within / (total_n - k)
    if not np.isfinite(ms_within):
        return float("nan")
    denom = sst + ms_within
    if denom == 0:
        return float("nan")
    return float((ss_between - (k - 1) * ms_within) / denom)


def holm_adjust(pvals: pd.Series) -> pd.Series:
    if pvals.size == 0:
        return pvals
    order = pvals.sort_values().index.tolist()
    adjusted: dict[tuple[str, str], float] = {}
    m = len(order)
    for position, key in enumerate(order):
        factor = m - position
        adjusted[key] = min(1.0, factor * float(pvals.loc[key]))
    for idx in range(1, len(order)):
        prev = order[idx - 1]
        curr = order[idx]
        adjusted[curr] = max(adjusted[curr], adjusted[prev])
    return pd.Series(adjusted)[pvals.index]



def games_howell_ln(frame: pd.DataFrame, group_col: str, value_col: str) -> pd.DataFrame:
    result = pg.pairwise_gameshowell(dv=value_col, between=group_col, data=frame).reset_index(drop=True)

    ci_lower = None
    ci_upper = None
    ci_candidates = [col for col in result.columns if 'CI95' in col.replace(' ', '').upper()]

    def _series_as_tuple(series: pd.Series) -> tuple[pd.Series, pd.Series] | None:
        try:
            tuples = series.apply(lambda v: (float(v[0]), float(v[1])))
        except Exception:
            return None
        lo = tuples.map(lambda pair: pair[0])
        hi = tuples.map(lambda pair: pair[1])
        return lo, hi

    for column in ci_candidates:
        label = column.lower()
        series = result[column]
        if any(token in label for token in ['_low', '_lower', '-']) or label.endswith('lo'):
            ci_lower = series.astype(float)
        elif any(token in label for token in ['_high', '_upper', '+']) or label.endswith('hi'):
            ci_upper = series.astype(float)

    if (ci_lower is None or ci_upper is None) and 'CI95%' in result.columns:
        parsed = _series_as_tuple(result['CI95%'])
        if parsed is not None:
            ci_lower, ci_upper = parsed

    if (ci_lower is None or ci_upper is None) and ci_candidates:
        fallback = _series_as_tuple(result[ci_candidates[0]])
        if fallback is not None:
            ci_lower, ci_upper = fallback

    if ci_lower is None or ci_upper is None:
        se_col = next((col for col in ['se', 'SE'] if col in result.columns), None)
        df_col = next((col for col in ['dof', 'df'] if col in result.columns), None)
        if se_col and df_col:
            diff = (result['mean(A)'] - result['mean(B)']).astype(float)
            se = result[se_col].astype(float)
            dof = result[df_col].astype(float)
            tcrit = dof.apply(lambda df: stats.t.ppf(0.975, df) if np.isfinite(df) and df > 0 else np.nan)
            ci_lower = diff - tcrit * se
            ci_upper = diff + tcrit * se
        else:
            raise KeyError('Unable to determine CI95% bounds from pingouin.pairwise_gameshowell output')

    diff = (result['mean(A)'] - result['mean(B)']).astype(float)
    out = pd.DataFrame({
        'i': result['A'].astype(str),
        'j': result['B'].astype(str),
        'diff_ln': diff,
        'ratio': np.exp(diff),
        'ratio_ci_lo': np.exp(ci_lower.astype(float)),
        'ratio_ci_hi': np.exp(ci_upper.astype(float)),
        'p_raw': result['pval'].astype(float),
    })
    indexed = out.set_index(['i', 'j'])
    indexed['p_holm'] = holm_adjust(indexed['p_raw']).astype(float)
    return indexed.reset_index()

def compact_letter_display(pairs: pd.DataFrame, alpha: float = 0.05) -> dict[str, str]:
    items = sorted(set(pairs["i"]).union(set(pairs["j"])))
    letters = {item: "" for item in items}
    significant = {(row.i, row.j) for row in pairs.itertuples(index=False) if row.p_holm < alpha}
    significant |= {(b, a) for (a, b) in significant}
    remaining = set(items)
    current = ord("A")
    while remaining:
        group = []
        for item in sorted(remaining):
            if all((item, other) not in significant for other in group):
                group.append(item)
        for item in group:
            letters[item] += chr(current)
        remaining -= set(group)
        current += 1
    return letters


In [ ]:
def analyze_latency_effects(codec: str, pattern: str) -> None:
    """
    Compute omnibus and pairwise effect sizes on ln(time) for a given codec×pattern.
    Uses Welch ANOVA (ω²) and Games–Howell with ratio + standardized effect size.
    """
    runs = load_latency_runs(DATA_DIR, codec, pattern)
    if runs.empty:
        return

    token = f"{pattern}_{codec}"
    order = sorted(runs["format"].unique())

    # --- Omnibus effect: Welch ANOVA + omega^2 on ln(time) ---
    groups = {fmt: runs.loc[runs["format"] == fmt, "ln_time"].to_numpy()
              for fmt in order}
    anova = welch_anova_ln(groups)
    omega = omega_sq_oneway_ln(groups)

    pd.DataFrame(
        [{
            "codec": codec,
            "pattern": pattern,
            "F": anova["F"],
            "df_between": anova["df_between"],
            "df_within": anova["df_within"],
            "pvalue": anova["pvalue"],
            "omega_sq": omega,  # omnibus effect size
        }]
    ).to_csv(TABLE_DIR / f"tab_effects_{token}_welch_anova.csv", index=False)

    # --- Pairwise Games–Howell: ratios + standardized effect size ---
    gh = games_howell_ln(runs[["format", "ln_time"]].copy(), "format", "ln_time")

    # Add Hedges g per pair on ln(time) as another effect size
    hedges_g = []
    for row in gh.itertuples(index=False):
        a = runs.loc[runs["format"] == row.i, "ln_time"].to_numpy(float)
        b = runs.loc[runs["format"] == row.j, "ln_time"].to_numpy(float)
        n1, n2 = len(a), len(b)
        if n1 < 2 or n2 < 2:
            hedges_g.append(np.nan)
            continue
        s1 = a.std(ddof=1)
        s2 = b.std(ddof=1)
        sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)
        if sp2 <= 0 or not np.isfinite(sp2):
            hedges_g.append(np.nan)
            continue
        sp = math.sqrt(sp2)
        d = (a.mean() - b.mean()) / sp
        J = 1.0 - 3.0 / (4.0 * (n1 + n2) - 9.0)  # small-sample correction
        hedges_g.append(J * d)

    gh["hedges_g"] = hedges_g  # standardized effect size on ln(time)
    gh.to_csv(TABLE_DIR / f"tab_effects_{token}_pairwise_gameshowell.csv",
              index=False)


In [ ]:
for codec in CODECS:
    for pattern in PATTERNS:
        analyze_latency_effects(codec, pattern)

# Diagnostic Summaries


In [9]:
def plot_qq_ln_times(frame: pd.DataFrame, meta: dict[str, str], outdir: Path) -> None:
    z = (frame["ln_time"] - frame["ln_time"].mean()) / frame["ln_time"].std(ddof=1)
    z_sorted = np.sort(z.values)
    n = len(z_sorted)
    theoretical, envelope_lo, envelope_hi = qq_envelope(n)
    statistic, p_value = stats.shapiro(frame["ln_time"].values)
    plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN))
    plt.fill_between(theoretical, envelope_lo, envelope_hi, alpha=0.2, label="95% envelope")
    plt.plot(theoretical, z_sorted, marker="o", linestyle="", markersize=3, label="sample")
    limit = np.nanmax(np.abs(np.concatenate([theoretical, z_sorted])))
    limit = float(np.ceil(limit * 1.05 * 10) / 10)
    plt.plot([-limit, limit], [-limit, limit], linestyle="--", linewidth=1)
    title = f"Q–Q ln(time): {meta['fmt']}/{meta['codec']}/{meta['pattern']} | n={n} | Shapiro p={p_value:.3g}"
    plt.title(title)
    plt.xlabel("Theoretical z (N(0,1))")
    plt.ylabel("Sample z (standardized ln time)")
    plt.tight_layout()
    filename = f"qq_{meta['fmt']}_{meta['codec']}_{meta['pattern']}.png"
    plt.savefig(outdir / filename, dpi=150)
    plt.close()


def plot_drift_ln_times(frame: pd.DataFrame, meta: dict[str, str], outdir: Path) -> None:
    x = frame["i"].values
    y = frame["ln_time"].values
    geometric = float(np.exp(y.mean()))
    rho1 = lag1_autocorr(y)
    plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN))
    plt.plot(x, y, marker="o", linestyle="-", linewidth=1)
    y_smooth = smooth_series(x, y)
    plt.plot(x, y_smooth, linewidth=2)
    plt.axhline(np.log(geometric), linestyle="--", linewidth=1)
    title = f"Run-order ln(time): {meta['fmt']}/{meta['codec']}/{meta['pattern']} | n={len(x)} | ρ₁={rho1:.2f}"
    plt.title(title)
    plt.xlabel("Repetition index")
    plt.ylabel("ln(time [s])")
    plt.tight_layout()
    filename = f"drift_{meta['fmt']}_{meta['codec']}_{meta['pattern']}.png"
    plt.savefig(outdir / filename, dpi=150)
    plt.close()


def build_run_summary(data_dir: Path, qq_dir: Path, drift_dir: Path) -> pd.DataFrame:
    fields = [
        "file",
        "format",
        "codec",
        "pattern",
        "run_id",
        "n",
        "geometric_mean_sec",
        "p95_sec",
        "shapiro_W",
        "shapiro_p",
        "rho1",
        "spearman_rho",
        "spearman_p",
    ]
    summaries: list[dict[str, object]] = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file() or not path.name.endswith(".json") or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        frame = load_runs(path)
        plot_qq_ln_times(frame, meta, qq_dir)
        plot_drift_ln_times(frame, meta, drift_dir)
        gm = float(np.exp(frame["ln_time"].mean()))
        p95 = float(np.exp(np.quantile(frame["ln_time"], 0.95)))
        shapiro_stat, shapiro_p = stats.shapiro(frame["ln_time"].values)
        rho1 = lag1_autocorr(frame["ln_time"].values)
        rho_s, p_s = stats.spearmanr(frame["i"].values, frame["ln_time"].values)
        summaries.append({
            "file": path.name,
            "format": meta["fmt"],
            "codec": meta["codec"],
            "pattern": meta["pattern"],
            "run_id": meta["runid"],
            "n": int(len(frame)),
            "geometric_mean_sec": gm,
            "p95_sec": p95,
            "shapiro_W": float(shapiro_stat),
            "shapiro_p": float(shapiro_p),
            "rho1": float(rho1),
            "spearman_rho": float(rho_s) if np.isfinite(rho_s) else np.nan,
            "spearman_p": float(p_s) if np.isfinite(p_s) else np.nan,
        })
    if not summaries:
        raise ValueError("No diagnostic summaries were produced.")
    summary = pd.DataFrame(summaries, columns=fields)
    summary = summary.sort_values(["format", "codec", "pattern", "run_id"]).reset_index(drop=True)
    summary["p95_over_gm"] = summary["p95_sec"] / summary["geometric_mean_sec"]
    summary["non_normal_flag"] = summary["shapiro_p"] < 0.05
    return summary


def format_run_summary_table(summary: pd.DataFrame) -> pd.DataFrame:
    if len(summary) == 0:
        raise ValueError("Summary table is empty.")
    table = summary.loc[:, [
        "format",
        "codec",
        "pattern",
        "n",
        "shapiro_p",
        "non_normal_flag",
        "rho1",
        "p95_over_gm",
        "spearman_p",
    ]].copy()
    table = table.rename(columns={
        "format": "Format",
        "codec": "Codec",
        "pattern": "Pattern",
        "n": "n",
        "shapiro_p": r"Shapiro $p$",
        "non_normal_flag": r"Non-normal (p<0.05)",
        "rho1": r"$\rho_1$",
        "p95_over_gm": r"$\mathrm{p95}/\mathrm{GM}$",
        "spearman_p": r"Spearman $p$",
    })
    table[r"Shapiro $p$"] = table[r"Shapiro $p$"].map(lambda x: f"{x:.3g}")
    table[r"Non-normal (p<0.05)"] = table[r"Non-normal (p<0.05)"].map(lambda v: r"\textbf{Yes}" if v else "No")
    table[r"$\rho_1$"] = table[r"$\rho_1$"].map(lambda x: f"{x:.2f}")
    table[r"$\mathrm{p95}/\mathrm{GM}$"] = table[r"$\mathrm{p95}/\mathrm{GM}$"].map(lambda x: f"{x:.2f}")
    table[r"Spearman $p$"] = table[r"Spearman $p$"].map(lambda x: "nan" if pd.isna(x) else f"{x:.3g}")
    table = table.sort_values(["Codec", "Pattern", "Format"]).reset_index(drop=True)
    return table


In [10]:
run_summary = build_run_summary(DATA_DIR, QQ_DIR, DRIFT_DIR)
if len(run_summary) == 0:
    raise ValueError("Diagnostic summary is empty.")
run_summary.to_csv(SUMMARY_DIR / "diagnostics_summary_full.csv", index=False)
run_summary_table = format_run_summary_table(run_summary)
run_summary_table.to_csv(SUMMARY_DIR / "diagnostics_summary_table.csv", index=False)


# Appendix Grids


In [11]:
def latest_runs(data_dir: Path) -> dict[tuple[str, str, str], tuple[str, Path]]:
    latest: dict[tuple[str, str, str], tuple[str, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or not path.name.endswith(".json") or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        run_id = meta["runid"]
        record = latest.get(key)
        if record is None or run_id > record[0]:
            latest[key] = (run_id, path)
    return latest


# def generate_appendix_grids(data_dir: Path, dest_dir: Path) -> None:
#     latest = latest_runs(data_dir)
#     cells_data: dict[tuple[str, str, str], dict[str, object]] = {}
#     for fmt in FORMATS:
#         for codec, pattern in ROWS:
#             key = (fmt, codec, pattern)
#             if key not in latest:
#                 continue
#             run_id, path = latest[key]
#             frame = load_runs(path)
#             cells_data[key] = {
#                 "frame": frame,
#                 "run_id": run_id,
#                 "shapiro_p": stats.shapiro(frame["ln_time"].values)[1],
#                 "rho1": lag1_autocorr(frame["ln_time"].values),
#             }
#     if not cells_data:
#         return
#     fig, axes = plt.subplots(len(ROWS), len(FORMATS), figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN), sharex=True, sharey=True)
#     for r, (codec, pattern) in enumerate(ROWS):
#         for c, fmt in enumerate(FORMATS):
#             ax = axes[r, c]
#             key = (fmt, codec, pattern)
#             if key not in cells_data:
#                 ax.axis("off")
#                 continue
#             frame = cells_data[key]["frame"]
#             z = (frame["ln_time"] - frame["ln_time"].mean()) / frame["ln_time"].std(ddof=1)
#             z_sorted = np.sort(z.values)
#             theoretical, envelope_lo, envelope_hi = qq_envelope(len(z_sorted))
#             ax.fill_between(theoretical, envelope_lo, envelope_hi, alpha=0.15)
#             ax.plot(theoretical, z_sorted, marker="o", linestyle="", markersize=2)
#             lim = 3.0
#             ax.plot([-lim, lim], [-lim, lim], linestyle="--", linewidth=0.8)
#             ax.set_xlim(-lim, lim)
#             ax.set_ylim(-lim, lim)
#             if r == 0:
#                 ax.set_title(fmt)
#             if c == 0:
#                 ax.set_ylabel(f"{codec}/{pattern}")
#             ax.text(0.98, 0.02, f"p={cells_data[key]['shapiro_p']:.3g}", transform=ax.transAxes, ha="right", va="bottom")
#     fig.suptitle("Q–Q plots of ln(time) by codec/pattern and format")
#     fig.text(0.5, 0.005, "Theoretical z (N(0,1))", ha="center")
#     fig.text(0.005, 0.5, "Sample z (standardized ln time)", va="center", rotation="vertical")
#     fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
#     plt.savefig(dest_dir / "appendix_QQ_grid_6x4.png", dpi=300)
#     plt.savefig(dest_dir / "appendix_QQ_grid_6x4.pdf")
#     plt.close(fig)
#     fig, axes = plt.subplots(len(ROWS), len(FORMATS), figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN), sharex=True, sharey=False)
#     for r, (codec, pattern) in enumerate(ROWS):
#         for c, fmt in enumerate(FORMATS):
#             ax = axes[r, c]
#             key = (fmt, codec, pattern)
#             if key not in cells_data:
#                 ax.axis("off")
#                 continue
#             frame = cells_data[key]["frame"]
#             x = frame["i"].values
#             y = frame["ln_time"].values
#             quantiles = np.quantile(y, [0.02, 0.98])
#             if not np.isfinite(quantiles).all() or quantiles[1] <= quantiles[0]:
#                 ymin, ymax = float(y.min()), float(y.max())
#             else:
#                 ymin, ymax = float(quantiles[0]), float(quantiles[1])
#             span = max(ymax - ymin, 1e-3)
#             pad = 0.05 * span
#             ax.set_ylim(ymin - pad, ymax + pad)
#             ax.plot(x, y, marker="o", linestyle="-", linewidth=0.8, markersize=2)
#             y_smooth = smooth_series(x, y)
#             ax.plot(x, y_smooth, linewidth=1.5)
#             geometric = float(np.exp(y.mean()))
#             ax.axhline(np.log(geometric), linestyle="--", linewidth=0.8)
#             if r == 0:
#                 ax.set_title(fmt)
#             if c == 0:
#                 ax.set_ylabel(f"{codec}/{pattern}")
#             if r == len(ROWS) - 1:
#                 ax.set_xlabel("rep")
#             ax.text(0.98, 0.02, f"ρ₁={cells_data[key]['rho1']:.2f}", transform=ax.transAxes, ha="right", va="bottom")
#     fig.suptitle("Run-order traces of ln(time) by codec/pattern and format")
#     fig.text(0.005, 0.5, "ln(time [s])", va="center", rotation="vertical")
#     fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
#     plt.savefig(dest_dir / "appendix_Drift_grid_6x4.png", dpi=300)
#     plt.savefig(dest_dir / "appendix_Drift_grid_6x4.pdf")
#     plt.close(fig)


In [12]:
SLICE_ROWS = [("gzip", "slice"), ("lz4", "slice"), ("zstd", "slice")]
FULL_ROWS  = [("gzip", "full"),  ("lz4", "full"),  ("zstd", "full")]

TEXTWIDTH_IN = 5.8          # approx \textwidth in inches
QQ_HEIGHT_IN = 6.5          # good for 3 rows
DRIFT_HEIGHT_IN = 6.5

def _plot_qq_grid(cells_data, rows, dest_path: Path, title: str) -> None:
    fig, axes = plt.subplots(
        len(rows), len(FORMATS),
        figsize=(TEXTWIDTH_IN, QQ_HEIGHT_IN),
        sharex=True, sharey=True,
    )

    for r, (codec, pattern) in enumerate(rows):
        for c, fmt in enumerate(FORMATS):
            ax = axes[r, c]
            key = (fmt, codec, pattern)
            if key not in cells_data:
                ax.axis("off")
                continue
            frame = cells_data[key]["frame"]
            z = (frame["ln_time"] - frame["ln_time"].mean()) / frame["ln_time"].std(ddof=1)
            z_sorted = np.sort(z.values)
            theoretical, envelope_lo, envelope_hi = qq_envelope(len(z_sorted))
            ax.fill_between(theoretical, envelope_lo, envelope_hi, alpha=0.15)
            ax.plot(theoretical, z_sorted, marker="o", linestyle="", markersize=2)
            lim = 3.0
            ax.plot([-lim, lim], [-lim, lim], linestyle="--", linewidth=0.8)
            ax.set_xlim(-lim, lim)
            ax.set_ylim(-lim, lim)
            if r == 0:
                ax.set_title(fmt)
            if c == 0:
                ax.set_ylabel(f"{codec}/{pattern}")
            ax.text(
                0.98, 0.02,
                f"p={cells_data[key]['shapiro_p']:.3g}",
                transform=ax.transAxes,
                ha="right", va="bottom",
            )

    fig.suptitle(title)
    fig.text(0.5, 0.01, "Theoretical z (N(0,1))", ha="center")
    fig.text(0.01, 0.5, "Sample z (standardized ln time)", va="center", rotation="vertical")
    fig.tight_layout(rect=(0.04, 0.04, 1, 0.95))
    fig.savefig(dest_path.with_suffix(".png"), dpi=300)
    fig.savefig(dest_path.with_suffix(".pdf"))
    plt.close(fig)

def _plot_drift_grid(cells_data, rows, dest_path: Path, title: str) -> None:
    fig, axes = plt.subplots(
        len(rows), len(FORMATS),
        figsize=(TEXTWIDTH_IN, DRIFT_HEIGHT_IN),
        sharex=True, sharey=False,
    )

    for r, (codec, pattern) in enumerate(rows):
        for c, fmt in enumerate(FORMATS):
            ax = axes[r, c]
            key = (fmt, codec, pattern)
            if key not in cells_data:
                ax.axis("off")
                continue
            frame = cells_data[key]["frame"]
            x = frame["i"].values
            y = frame["ln_time"].values
            quantiles = np.quantile(y, [0.02, 0.98])
            if not np.isfinite(quantiles).all() or quantiles[1] <= quantiles[0]:
                ymin, ymax = float(y.min()), float(y.max())
            else:
                ymin, ymax = float(quantiles[0]), float(quantiles[1])
            span = max(ymax - ymin, 1e-3)
            pad = 0.05 * span
            ax.set_ylim(ymin - pad, ymax + pad)
            ax.plot(x, y, marker="o", linestyle="-", linewidth=0.8, markersize=2)
            y_smooth = smooth_series(x, y)
            ax.plot(x, y_smooth, linewidth=1.5)
            geometric = float(np.exp(y.mean()))
            ax.axhline(np.log(geometric), linestyle="--", linewidth=0.8)
            if r == 0:
                ax.set_title(fmt)
            if c == 0:
                ax.set_ylabel(f"{codec}/{pattern}")
            if r == len(rows) - 1:
                ax.set_xlabel("rep")
            ax.text(
                0.98, 0.02,
                f"ρ₁={cells_data[key]['rho1']:.2f}",
                transform=ax.transAxes,
                ha="right", va="bottom",
            )

    fig.suptitle(title)
    fig.text(0.01, 0.5, "ln(time [s])", va="center", rotation="vertical")
    fig.tight_layout(rect=(0.04, 0.04, 1, 0.95))
    fig.savefig(dest_path.with_suffix(".png"), dpi=300)
    fig.savefig(dest_path.with_suffix(".pdf"))
    plt.close(fig)

def generate_appendix_grids(data_dir: Path, dest_dir: Path) -> None:
    latest = latest_runs(data_dir)
    cells_data: dict[tuple[str, str, str], dict[str, object]] = {}

    for fmt in FORMATS:
        for codec, pattern in SLICE_ROWS + FULL_ROWS:
            key = (fmt, codec, pattern)
            if key not in latest:
                continue
            run_id, path = latest[key]
            frame = load_runs(path)
            cells_data[key] = {
                "frame": frame,
                "run_id": run_id,
                "shapiro_p": stats.shapiro(frame["ln_time"].values)[1],
                "rho1": lag1_autocorr(frame["ln_time"].values),
            }
    if not cells_data:
        return

    # Q–Q grids: slices and full
    _plot_qq_grid(cells_data, SLICE_ROWS, dest_dir / "appendix_QQ_grid_slice_3x4",
                  "Q–Q plots of ln(time) — slices by codec and format")
    _plot_qq_grid(cells_data, FULL_ROWS, dest_dir / "appendix_QQ_grid_full_3x4",
                  "Q–Q plots of ln(time) — full scans by codec and format")

    # Drift grids: slices and full
    _plot_drift_grid(cells_data, SLICE_ROWS, dest_dir / "appendix_Drift_grid_slice_3x4",
                     "Run-order traces of ln(time) — slices by codec and format")
    _plot_drift_grid(cells_data, FULL_ROWS, dest_dir / "appendix_Drift_grid_full_3x4",
                     "Run-order traces of ln(time) — full scans by codec and format")


In [13]:
generate_appendix_grids(DATA_DIR, DIAGNOSTICS_DIR)

# Request Intensity


In [14]:
def latest_run_by_mtime(data_dir: Path) -> dict[tuple[str, str, str], Path]:
    latest: dict[tuple[str, str, str], tuple[float, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "cloudwatch" in path.name:
            continue
        meta = parse_run_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        record = latest.get(key)
        mtime = path.stat().st_mtime
        if record is None or mtime > record[0]:
            latest[key] = (mtime, path)
    return {key: value for key, (_, value) in latest.items()}


def latest_cloudwatch_by_mtime(data_dir: Path) -> dict[tuple[str, str, str], Path]:
    latest: dict[tuple[str, str, str], tuple[float, Path]] = {}
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "_cloudwatch" not in path.name:
            continue
        meta = parse_cloudwatch_meta(path.name)
        if not meta:
            continue
        key = (meta["fmt"], meta["codec"], meta["pattern"])
        record = latest.get(key)
        mtime = path.stat().st_mtime
        if record is None or mtime > record[0]:
            latest[key] = (mtime, path)
    return {key: value for key, (_, value) in latest.items()}


def compute_request_intensity_table(data_dir: Path) -> pd.DataFrame:
    run_files = latest_run_by_mtime(data_dir)
    cloudwatch_files = latest_cloudwatch_by_mtime(data_dir)
    rows: list[dict[str, object]] = []
    for key in sorted(set(run_files) & set(cloudwatch_files)):
        fmt, codec, pattern = key
        with open(run_files[key], "r") as handle:
            run_payload = json.load(handle)
        n_included = int(len(run_payload["data"]))
        with open(cloudwatch_files[key], "r") as handle:
            cw_payload = json.load(handle)
        gets = float(cw_payload.get("GetRequestsSum", np.nan))
        bytes_downloaded = float(cw_payload.get("BytesDownloadedSum", np.nan))
        gib = bytes_downloaded / (2 ** 30) if np.isfinite(bytes_downloaded) else np.nan
        gets_per_rep = gets / n_included if n_included > 0 else np.nan
        gets_per_gib = gets / gib if np.isfinite(gib) and gib > 0 else np.nan
        rep_lo, rep_hi = garwood_rate_ci(gets, n_included)
        gib_lo, gib_hi = garwood_rate_ci(gets, gib)
        rows.append({
            "format": fmt,
            "codec": codec,
            "pattern": pattern,
            "GetRequestsSum": gets,
            "BytesDownloadedSum_GiB": gib,
            "n_included": n_included,
            "GETs_per_rep": gets_per_rep,
            "GETs_per_rep_lo": rep_lo,
            "GETs_per_rep_hi": rep_hi,
            "GETs_per_GiB": gets_per_gib,
            "GETs_per_GiB_lo": gib_lo,
            "GETs_per_GiB_hi": gib_hi,
            "run_file": run_files[key].name,
            "cw_file": cloudwatch_files[key].name,
        })
    frame = pd.DataFrame(rows)
    frame = frame.sort_values(["codec", "pattern", "format"]).reset_index(drop=True)
    return frame


def plot_request_intensity_series(frame: pd.DataFrame, metric: str, lower: str, upper: str, prefix: str, ylabel: str) -> None:
    for codec in CODECS:
        for pattern in PATTERNS:
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].copy()
            subset = subset.set_index("format").reindex(FORMATS).dropna(subset=[metric]).reset_index()
            x = np.arange(len(subset))
            values = subset[metric].to_numpy(float)
            lower_bounds = np.maximum(0.0, subset[lower].to_numpy(float))
            upper_bounds = np.maximum(0.0, subset[upper].to_numpy(float))
            errors = [np.maximum(0.0, values - lower_bounds), np.maximum(0.0, upper_bounds - values)]
            ymin = float(np.nanmin(np.minimum(lower_bounds, values)))
            ymax = float(np.nanmax(np.maximum(upper_bounds, values)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.08, floor=0.0)
            plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN))
            ax = plt.gca()
            ax.errorbar(x, values, yerr=errors, fmt="o", capsize=4, linewidth=1)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            ax.set_ylim(lo, hi)
            ax.set_ylabel(ylabel)
            ax.set_title(f"Request intensity — {codec}/{pattern}")
            ax.yaxis.grid(True, alpha=0.3)
            for xi, yi in zip(x, values):
                ax.text(xi, yi, f"{yi:.3g}", ha="center", va="bottom")
            plt.tight_layout()
            plt.savefig(FIGURE_DIR / f"{prefix}_{codec}_{pattern}.png", dpi=180)
            plt.close()


def plot_request_intensity_grid(frame: pd.DataFrame, metric: str, lower: str, upper: str, ylabel: str, prefix: str, minimum_ci_fraction: float = 0.003) -> None:
    fig, axes = plt.subplots(nrows=len(CODECS), ncols=len(PATTERNS), figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN), sharex=False, sharey=False)
    for r, codec in enumerate(CODECS):
        for c, pattern in enumerate(PATTERNS):
            ax = axes[r, c] if len(CODECS) > 1 else axes[c]
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].set_index("format").reindex(FORMATS).reset_index()
            subset = subset.dropna(subset=[metric])
            ax.set_title(f"{codec} / {pattern}")
            x = np.arange(len(subset))
            values = subset[metric].to_numpy(float)
            lower_bounds = np.maximum(0.0, subset[lower].to_numpy(float))
            upper_bounds = np.maximum(0.0, subset[upper].to_numpy(float))
            err_low_true = np.maximum(0.0, values - lower_bounds)
            err_high_true = np.maximum(0.0, upper_bounds - values)
            epsilon = np.maximum(minimum_ci_fraction * np.maximum(values, 1.0), 1e-9)
            err_low = np.maximum(err_low_true, epsilon)
            err_high = np.maximum(err_high_true, epsilon)
            text_positions = [yi + err_hi + 0.01 * np.maximum(yi, 1.0) for yi, err_hi in zip(values, err_high)]
            ymin = float(np.nanmin(np.minimum(lower_bounds, values)))
            ymax = float(np.nanmax(np.maximum(upper_bounds, values)))
            if text_positions:
                ymax = max(ymax, float(np.nanmax(text_positions)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.15, floor=0.0)
            ax.errorbar(x, values, yerr=[err_low, err_high], fmt="o", markersize=4, markeredgewidth=0.8, elinewidth=1.6, capsize=5)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            ax.set_ylim(lo, hi)
            ax.yaxis.grid(True, alpha=0.35)
            if c == 0:
                ax.set_ylabel(ylabel)
            for xi, yi, text_y in zip(x, values, text_positions):
                clipped = min(text_y, hi - 0.02 * (hi - lo))
                if xi == 0 and len(x) > 1:
                    align = "left"
                elif xi == len(x) - 1 and len(x) > 1:
                    align = "right"
                else:
                    align = "center"
                ax.text(xi, clipped, f"{yi:.3g}", ha=align, va="bottom", clip_on=True)
    fig.suptitle(ylabel)
    fig.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.savefig(FIGURE_DIR / f"{prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{prefix}.pdf")
    plt.close(fig)


In [15]:
request_intensity = compute_request_intensity_table(DATA_DIR)
if len(request_intensity) == 0:
    raise ValueError("Request intensity table is empty.")
request_intensity.to_csv(TABLE_DIR / "tab_5_4_request_intensity.csv", index=False)
plot_request_intensity_series(
    request_intensity,
    metric="GETs_per_rep",
    lower="GETs_per_rep_lo",
    upper="GETs_per_rep_hi",
    prefix="fig_5_4_reqintensity_perRep_point_ci",
    ylabel="GETs per repetition",
)
plot_request_intensity_series(
    request_intensity,
    metric="GETs_per_GiB",
    lower="GETs_per_GiB_lo",
    upper="GETs_per_GiB_hi",
    prefix="fig_5_4_reqintensity_perGiB_point_ci",
    ylabel="GETs per GiB",
)
plot_request_intensity_grid(
    request_intensity,
    metric="GETs_per_rep",
    lower="GETs_per_rep_lo",
    upper="GETs_per_rep_hi",
    ylabel="GETs per repetition",
    prefix="fig_5_4_reqintensity_perRep_grid",
    minimum_ci_fraction=0.003,
)
plot_request_intensity_grid(
    request_intensity,
    metric="GETs_per_GiB",
    lower="GETs_per_GiB_lo",
    upper="GETs_per_GiB_hi",
    ylabel="GETs per GiB",
    prefix="fig_5_4_reqintensity_perGiB_grid",
    minimum_ci_fraction=0.003,
)


In [16]:
def plot_request_intensity_box_by_format(frame: pd.DataFrame, prefix: str) -> None:
    """
    Merged figure: two rows × one column of boxplots grouped by format.

    Top panel: GETs per repetition.
    Bottom panel: GETs per GiB.

    Each box aggregates values across codecs and patterns for a given format.
    """
    # Ensure formats are ordered consistently
    formats = list(FORMATS)

    # Prepare data per metric
    metrics = [
        ("GETs_per_rep", "GETs per repetition"),
        ("GETs_per_GiB", "GETs per GiB"),
    ]

    fig, axes = plt.subplots(
        nrows=len(metrics),
        ncols=1,
        figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN),
        sharex=True,
    )
    axes = np.atleast_1d(axes)

    for ax, (metric, ylabel) in zip(axes, metrics):
        data_per_format: list[np.ndarray] = []
        for fmt in formats:
            sub = frame[frame["format"] == fmt][metric].to_numpy(dtype=float)
            sub = sub[np.isfinite(sub)]
            if sub.size == 0:
                # boxplot cannot handle empty data; put nan placeholder
                sub = np.array([np.nan])
            data_per_format.append(sub)

        positions = np.arange(1, len(formats) + 1)

        bp = ax.boxplot(
            data_per_format,
            positions=positions,
            widths=0.6,
            patch_artist=True,
            manage_ticks=False,
        )

        # Simple coloring to improve readability
        for patch in bp["boxes"]:
            patch.set_facecolor("lightgray")
            patch.set_alpha(0.8)

        ax.set_ylabel(ylabel)
        ax.yaxis.grid(True, alpha=0.3)

        ax.set_xticks(positions)
        ax.set_xticklabels(formats)

    axes[-1].set_xlabel("Format")
    fig.suptitle("S3 request intensity by format (aggregated across codecs and patterns)")
    fig.tight_layout(rect=[0.03, 0.00, 1.0, 0.95])

    plt.savefig(FIGURE_DIR / f"{prefix}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{prefix}.pdf", bbox_inches="tight")

    graphs_dir = Path("figures/graphs")
    graphs_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(graphs_dir / f"{prefix}.pdf", bbox_inches="tight")

    plt.close(fig)


In [17]:
request_intensity = compute_request_intensity_table(DATA_DIR)
if len(request_intensity) == 0:
    raise ValueError("Request intensity table is empty.")
request_intensity.to_csv(TABLE_DIR / "tab_5_4_request_intensity.csv", index=False)

# New merged figure (main text)
plot_request_intensity_box_by_format(
    request_intensity,
    prefix="fig_reqintensity_box_by_format",
)

In [19]:
from pathlib import Path

def plot_codec_summary_by_format(out_prefix: str = "fig_codec_summary_by_format") -> None:
    """
    Two-panel figure, grouped by format:
      - Top: GM slice latency (ms) with dependence-robust 95% CIs.
      - Bottom: GM full-scan throughput (GiB/s) with transformed 95% CIs.

    Each format has three bars (gzip, LZ4, zstd).
    """
    tables_path = Path("tables")  # adjust if needed

    # --- Slice GM latency (ms) per format×codec ---
    slice_rows = []
    for codec in CODECS:
        token = f"slice_{codec}"
        path = tables_path / f"tab_mbb_{token}_latency.csv"
        df = pd.read_csv(path)
        df = df.rename(columns={c: c.strip() for c in df.columns})
        if "Format" in df.columns:
            df = df.rename(columns={"Format": "format"})
        df["codec"] = codec
        slice_rows.append(df[["format", "codec", "GM_latency_ms", "GM_ci_lo_ms", "GM_ci_hi_ms"]])

    slice_df = pd.concat(slice_rows, ignore_index=True)
    slice_df["format"] = pd.Categorical(slice_df["format"], categories=FORMATS, ordered=True)
    slice_df = slice_df.sort_values(["format", "codec"]).reset_index(drop=True)

    # --- Full-scan GM throughput (GiB/s) per format×codec ---
    full_path = tables_path / "tab_cross_full_throughput_vs_GETsGiB.csv"
    full_df = pd.read_csv(full_path)
    full_df = full_df.rename(columns={c: c.strip() for c in full_df.columns})
    full_df = full_df[["format", "codec", "GM_throughput_GiB_s", "GM_throughput_lo", "GM_throughput_hi"]].copy()
    full_df["format"] = pd.Categorical(full_df["format"], categories=FORMATS, ordered=True)
    full_df = full_df.sort_values(["format", "codec"]).reset_index(drop=True)

    # --- Plot: two panels stacked ---
    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(TEXTWIDTH_IN, 6.5),   # adjust height to taste
        sharex=True,
    )

    # Common positions per format, and offsets per codec
    x = np.arange(len(FORMATS))
    width = 0.22
    offsets = {
        CODECS[0]: -width,
        CODECS[1]: 0.0,
        CODECS[2]: +width,
    }
    colors = {
        "gzip": "C0",
        "lz4": "C1",
        "zstd": "C2",
    }

    # --- Top panel: slice GM latency ---
    ax_top = axes[0]
    for codec in CODECS:
        sub = slice_df[slice_df["codec"] == codec].set_index("format").reindex(FORMATS).reset_index()
        gm = sub["GM_latency_ms"].to_numpy(float)
        lo = sub["GM_ci_lo_ms"].to_numpy(float)
        hi = sub["GM_ci_hi_ms"].to_numpy(float)
        err_low = np.maximum(0.0, gm - lo)
        err_high = np.maximum(0.0, hi - gm)
        xpos = x + offsets[codec]

        ax_top.bar(xpos, gm, width=width, color=colors.get(codec, None), alpha=0.8, label=codec)
        ax_top.errorbar(xpos, gm, yerr=np.vstack([err_low, err_high]),
                        fmt="none", ecolor="black", elinewidth=1.0, capsize=3)

    ax_top.set_ylabel("GM slice latency (ms)")
    ax_top.set_title("Codec comparison by format — slice latency")
    ax_top.yaxis.grid(True, alpha=0.3)
    ax_top.legend(title="Codec", loc="upper right")

    # --- Bottom panel: full GM throughput ---
    ax_bot = axes[1]
    for codec in CODECS:
        sub = full_df[full_df["codec"] == codec].set_index("format").reindex(FORMATS).reset_index()
        thr = sub["GM_throughput_GiB_s"].to_numpy(float)
        lo = sub["GM_throughput_lo"].to_numpy(float)
        hi = sub["GM_throughput_hi"].to_numpy(float)
        err_low = np.maximum(0.0, thr - lo)
        err_high = np.maximum(0.0, hi - thr)
        xpos = x + offsets[codec]

        ax_bot.bar(xpos, thr, width=width, color=colors.get(codec, None), alpha=0.8, label=codec)
        ax_bot.errorbar(xpos, thr, yerr=np.vstack([err_low, err_high]),
                        fmt="none", ecolor="black", elinewidth=1.0, capsize=3)

    ax_bot.set_ylabel("GM full-scan throughput (GiB/s)")
    ax_bot.set_xlabel("Format")
    ax_bot.set_xticks(x)
    ax_bot.set_xticklabels(FORMATS)
    ax_bot.yaxis.grid(True, alpha=0.3)
    ax_bot.set_title("Codec comparison by format — full-scan throughput")

    fig.suptitle("Codec comparison by format (slice latency and full-scan throughput)")
    fig.tight_layout(rect=[0.03, 0.03, 1.0, 0.95])

    # Save
    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf", bbox_inches="tight")

    graphs_dir = Path("figures/graphs")
    graphs_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(graphs_dir / f"{out_prefix}.pdf", bbox_inches="tight")

    plt.close(fig)


In [20]:
plot_codec_summary_by_format(out_prefix="fig_codec_summary_by_format")

# CloudWatch Latencies


In [23]:
def aggregate_cloudwatch_latencies(data_dir: Path) -> pd.DataFrame:
    records: list[dict[str, object]] = []
    for path in data_dir.iterdir():
        if not path.is_file() or path.suffix != ".json" or "_cloudwatch" not in path.name:
            continue
        meta = parse_cloudwatch_meta(path.name)
        if not meta:
            continue
        with open(path, "r") as handle:
            payload = json.load(handle)
        records.append({
            "format": meta["fmt"],
            "codec": meta["codec"],
            "pattern": meta["pattern"],
            "GetRequestsSum": float(payload.get("GetRequestsSum", np.nan)),
            "BytesDownloadedSum": float(payload.get("BytesDownloadedSum", np.nan)),
            "FirstByteLatencyAverage": float(payload.get("FirstByteLatencyAverage", np.nan)),
            "FirstByteLatencyP95": float(payload.get("FirstByteLatencyP95", np.nan)),
            "TotalRequestLatencyAverage": float(payload.get("TotalRequestLatencyAverage", np.nan)),
            "TotalRequestLatencyP95": float(payload.get("TotalRequestLatencyP95", np.nan)),
            "source": path.name,
        })
    sessions = pd.DataFrame(records)
    if len(sessions) == 0:
        raise ValueError("CloudWatch latency sessions are empty.")
    def reducer(group: pd.DataFrame) -> pd.Series:
        weights = group["GetRequestsSum"].to_numpy(float)
        weighted_sum = float(np.nansum(weights)) if np.isfinite(np.nansum(weights)) else 0.0
        def weighted_average(series: pd.Series) -> float:
            if weighted_sum > 0 and np.isfinite(series).all():
                return float(np.average(series.to_numpy(float), weights=weights))
            return float(series.mean())
        return pd.Series({
            "sessions": int(len(group)),
            "GetRequestsSum_total": float(group["GetRequestsSum"].sum()),
            "BytesDownloadedSum_total": float(group["BytesDownloadedSum"].sum()),
            "FirstByteLatencyAverage_ms": weighted_average(group["FirstByteLatencyAverage"]),
            "FirstByteLatencyP95_ms": float(group["FirstByteLatencyP95"].mean()),
            "TotalRequestLatencyAverage_ms": weighted_average(group["TotalRequestLatencyAverage"]),
            "TotalRequestLatencyP95_ms": float(group["TotalRequestLatencyP95"].mean()),
        })
    aggregated = sessions.groupby(["format", "codec", "pattern"]).apply(reducer).reset_index()
    aggregated = aggregated.sort_values(["codec", "pattern", "format"]).reset_index(drop=True)
    rounded = aggregated.copy()
    for column in [
        "FirstByteLatencyAverage_ms",
        "FirstByteLatencyP95_ms",
        "TotalRequestLatencyAverage_ms",
        "TotalRequestLatencyP95_ms",
    ]:
        rounded[column] = rounded[column].map(lambda x: np.nan if pd.isna(x) else round(float(x), 3))
    rounded.to_csv(TABLE_DIR / "tab_5_5_cw_latencies.csv", index=False)
    return aggregated


def plot_latency_grid(frame: pd.DataFrame, average_col: str, p95_col: str, title: str, prefix: str, ylabel: str) -> None:
    fig, axes = plt.subplots(nrows=len(CODECS), ncols=len(PATTERNS), figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN), sharex=False, sharey=False)
    width = 0.35
    for r, codec in enumerate(CODECS):
        for c, pattern in enumerate(PATTERNS):
            ax = axes[r, c] if len(CODECS) > 1 else axes[c]
            subset = frame[(frame["codec"] == codec) & (frame["pattern"] == pattern)].set_index("format").reindex(FORMATS).reset_index()
            subset = subset.dropna(subset=[average_col, p95_col])
            if len(subset) == 0:
                raise ValueError(f"Missing latency data for {codec}/{pattern}.")
            ax.set_title(f"{codec} / {pattern}")
            x = np.arange(len(subset))
            avg_vals = subset[average_col].to_numpy(float)
            p95_vals = subset[p95_col].to_numpy(float)
            ax.bar(x - width / 2, avg_vals, width, label="avg")
            ax.bar(x + width / 2, p95_vals, width, label="p95")
            ymin = float(np.nanmin(np.minimum(avg_vals, p95_vals)))
            ymax = float(np.nanmax(np.maximum(avg_vals, p95_vals)))
            lo, hi = nice_ylim(ymin, ymax, pad_frac=0.12, floor=0.0)
            ax.set_ylim(lo, hi)
            ax.set_xticks(x)
            ax.set_xticklabels(subset["format"])
            if c == 0:
                ax.set_ylabel(ylabel)
            ax.yaxis.grid(True, alpha=0.35)
            for xi, yi in zip(x - width / 2, avg_vals):
                ax.text(xi, yi, f"{yi:.1f}", ha="center", va="bottom", fontsize=6)
            for xi, yi in zip(x + width / 2, p95_vals):
                ax.text(xi, yi, f"{yi:.1f}", ha="center", va="bottom", fontsize=6)
            if r == 0 and c == len(PATTERNS) - 1:
                ax.legend(loc="upper right")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{prefix}.pdf")
    plt.close(fig)


In [24]:
cw_aggregated = aggregate_cloudwatch_latencies(DATA_DIR)
if len(cw_aggregated) == 0:
    raise ValueError("CloudWatch aggregate table is empty.")
plot_latency_grid(
    cw_aggregated,
    average_col="FirstByteLatencyAverage_ms",
    p95_col="FirstByteLatencyP95_ms",
    title="S3 FirstByteLatency: avg vs p95",
    prefix="fig_5_5_firstbyte_grid",
    ylabel="FirstByte latency (ms)",
)
plot_latency_grid(
    cw_aggregated,
    average_col="TotalRequestLatencyAverage_ms",
    p95_col="TotalRequestLatencyP95_ms",
    title="S3 TotalRequestLatency: avg vs p95",
    prefix="fig_5_5_total_grid",
    ylabel="TotalRequest latency (ms)",
)


/var/folders/v7/xx2vbnj149dcn0y3fyq07fww0000gn/T/ipykernel_15502/3103121122.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aggregated = sessions.groupby(["format", "codec", "pattern"]).apply(reducer).reset_index()


# Latency Analysis


In [ ]:
# def analyze_latency(codec: str, pattern: str) -> None:
#     runs = load_latency_runs(DATA_DIR, codec, pattern)
#     order_candidates = runs["format"].unique()
#     if len(order_candidates) == 0:
#         raise ValueError(f"No runs available for {codec}/{pattern}.")

#     token = f"{pattern}_{codec}"
#     base_seed = RNG_SEED + 97 * CODECS.index(codec) + 13 * PATTERNS.index(pattern)

#     summary_rows = []
#     for fmt, group in runs.groupby("format"):
#         ln_values = group["ln_time"].to_numpy()
#         latency_ms = 1000.0 * group["time_sec"].to_numpy()
#         gm, gm_lo, gm_hi = gmean_ci_from_ln(ln_values)
#         p95, p95_lo, p95_hi = bootstrap_quantile(
#             latency_ms,
#             q=0.95,
#             draws=BOOTSTRAP_DRAWS,
#             seed=base_seed + FORMATS.index(fmt),
#         )
#         summary_rows.append({
#             "format": fmt,
#             "n": len(group),
#             "gm_ms": 1000.0 * gm,
#             "gm_lo_ms": 1000.0 * gm_lo,
#             "gm_hi_ms": 1000.0 * gm_hi,
#             "p95_ms": p95,
#             "p95_lo_ms": p95_lo,
#             "p95_hi_ms": p95_hi,
#         })

#     summary_df = pd.DataFrame(summary_rows)
#     sorted_by_mean = summary_df.sort_values("gm_ms", ascending=True).reset_index(drop=True)
#     order = sorted_by_mean["format"].tolist()

#     summary_df.sort_values(["format"]).to_csv(
#         TABLE_DIR / f"tab_5_6a_{token}_latency.csv",
#         index=False,
#     )

#     x_positions = np.arange(len(sorted_by_mean))
#     gm_values = sorted_by_mean["gm_ms"].to_numpy(float)
#     ci_lo = sorted_by_mean["gm_lo_ms"].to_numpy(float)
#     ci_hi = sorted_by_mean["gm_hi_ms"].to_numpy(float)
#     yerr = np.vstack([gm_values - ci_lo, ci_hi - gm_values])

#     plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
#     plt.errorbar(
#         x_positions,
#         gm_values,
#         yerr=yerr,
#         fmt="o",
#         capsize=5,
#         elinewidth=1.6,
#     )
#     plt.xticks(x_positions, sorted_by_mean["format"])
#     plt.ylabel("Geometric mean latency (ms)")
#     plt.title(f"{pattern} latency — geometric mean ±95% CI, {codec}")
#     plt.gca().yaxis.grid(True, alpha=0.35)
#     plt.tight_layout()
#     plt.savefig(FIGURE_DIR / f"fig_5_6a_{token}_mean_ci_ms.png", dpi=220)
#     plt.savefig(FIGURE_DIR / f"fig_5_6a_{token}_mean_ci_ms.pdf")
#     plt.close()

#     runs_centered = runs.copy()
#     runs_centered["ln_resid"] = (
#         runs_centered["ln_time"]
#         - runs_centered.groupby("format")["ln_time"].transform("mean")
#     )

#     plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
#     data_resid = [
#         runs_centered.loc[runs_centered["format"] == fmt, "ln_resid"].to_numpy()
#         for fmt in order
#     ]
#     plt.violinplot(data_resid, showmeans=False, showextrema=False, showmedians=False)
#     rng = np.random.default_rng(base_seed)
#     for idx, fmt in enumerate(order, start=1):
#         points = runs_centered.loc[runs_centered["format"] == fmt, "ln_resid"].to_numpy()
#         jitter = rng.uniform(-0.08, 0.08, size=len(points))
#         plt.plot(idx + jitter, points, "o", markersize=2.2, alpha=0.6)
#     plt.axhline(0.0, linestyle="--", linewidth=1)
#     plt.ylim(-0.25, 0.25)
#     plt.xticks(range(1, len(order) + 1), order)
#     plt.ylabel("Centered ln(time) [s] (ln t − group mean)")
#     plt.title(f"{pattern} latency — centered dispersion, {codec}")
#     plt.tight_layout()
#     plt.savefig(FIGURE_DIR / f"fig_5_6b_{token}_centered_violin_ln.png", dpi=220)
#     plt.savefig(FIGURE_DIR / f"fig_5_6b_{token}_centered_violin_ln.pdf")
#     plt.close()

#     plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
#     for fmt in order:
#         series = 1000.0 * runs.loc[runs["format"] == fmt, "time_sec"].to_numpy()
#         x = np.sort(series)
#         y = np.arange(1, len(x) + 1) / len(x)
#         plt.plot(x, y, label=fmt)
#     plt.xlabel("Latency (ms)")
#     plt.ylabel("ECDF")
#     plt.title(f"{pattern} latency — ECDF, {codec}")
#     plt.legend()
#     plt.tight_layout()
#     plt.savefig(FIGURE_DIR / f"fig_5_6c_{token}_ecdf_ms.png", dpi=200)
#     plt.savefig(FIGURE_DIR / f"fig_5_6c_{token}_ecdf_ms.pdf")
#     plt.close()

#     groups = {fmt: runs.loc[runs["format"] == fmt, "ln_time"].to_numpy() for fmt in order}
#     anova = welch_anova_ln(groups)
#     omega = omega_sq_oneway_ln(groups)
#     pd.DataFrame([
#         {
#             "codec": codec,
#             "pattern": pattern,
#             "F": anova["F"],
#             "df_between": anova["df_between"],
#             "df_within": anova["df_within"],
#             "pvalue": anova["pvalue"],
#             "omega_sq": omega,
#         }
#     ]).to_csv(TABLE_DIR / f"tab_5_6b_{token}_welch_anova.csv", index=False)

#     gh = games_howell_ln(runs[["format", "ln_time"]].copy(), "format", "ln_time")
#     gh.to_csv(TABLE_DIR / f"tab_5_6c_{token}_pairwise_gameshowell.csv", index=False)
#     letters = compact_letter_display(gh)
#     pd.DataFrame({
#         "format": list(letters.keys()),
#         "letters": list(letters.values()),
#     }).sort_values("format").to_csv(
#         TABLE_DIR / f"tab_5_6c_{token}_letters.csv",
#         index=False,
#     )

#     index_map = {fmt: pos for pos, fmt in enumerate(order)}
#     matrix = np.ones((len(order), len(order)), dtype=float)
#     for row in gh.itertuples(index=False):
#         i_pos = index_map[row.i]
#         j_pos = index_map[row.j]
#         matrix[i_pos, j_pos] = row.ratio
#         matrix[j_pos, i_pos] = 1.0 / row.ratio if row.ratio != 0 else np.nan
#     plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
#     im = plt.imshow(matrix, aspect="auto", interpolation="nearest")
#     plt.xticks(range(len(order)), order, rotation=45, ha="right")
#     plt.yticks(range(len(order)), order)
#     plt.colorbar(im, fraction=0.046, pad=0.04)
#     plt.title(f"Pairwise time ratios (row / column) — {codec} {pattern}\n<1 means row faster")
#     for i in range(len(order)):
#         for j in range(len(order)):
#             value = matrix[i, j]
#             label = "nan" if not np.isfinite(value) else f"{value:.2f}"
#             plt.text(j, i, label, ha="center", va="center")
#     plt.tight_layout()
#     plt.savefig(FIGURE_DIR / f"fig_5_6d_{token}_ratio_heatmap.png", dpi=200)
#     plt.savefig(FIGURE_DIR / f"fig_5_6d_{token}_ratio_heatmap.pdf")
#     plt.close()

# for codec in CODECS:
#     for pattern in PATTERNS:
#         analyze_latency(codec, pattern)
# # analyze_latency("gzip", "slice")

# Latency Analysis New

In [31]:
from typing import Iterable

REF_FORMAT = "zarr"                 # reference for ratios
BLOCK_LENGTHS = (4, 6, 8)             # MBB block-length grid (n=50)
B = BOOTSTRAP_DRAWS
BASE_SEED = RNG_SEED + 97 * CODECS.index("gzip") + 13 * PATTERNS.index("slice")


def _mbb_resample(arr: np.ndarray, block_len: int, rng: np.random.Generator) -> np.ndarray:
    """Moving-block bootstrap with circular blocks."""
    n = len(arr)
    if n == 0:
        return arr
    if block_len <= 1:
        return rng.choice(arr, size=n, replace=True)
    ext = np.concatenate([arr, arr[: block_len - 1]])
    out = np.empty(n, dtype=float)
    pos = 0
    while pos < n:
        start = int(rng.integers(0, n))
        blk = ext[start : start + block_len]
        k = min(block_len, n - pos)
        out[pos : pos + k] = blk[:k]
        pos += k
    return out


def mbb_ci_mean_log(ln_values: np.ndarray, draws: int, block_lengths: Iterable[int], seed: int) -> tuple[float, float, float]:
    """GM point and 95% CI via MBB on ln(time); choose widest CI in log-space."""
    ln_values = np.asarray(ln_values, dtype=float)
    ln_values = ln_values[np.isfinite(ln_values)]
    if ln_values.size == 0:
        return np.nan, np.nan, np.nan
    point_ln = float(ln_values.mean())
    rng = np.random.default_rng(seed)
    candidates = []
    for L in block_lengths:
        means = np.empty(draws, dtype=float)
        for b in range(draws):
            boot = _mbb_resample(ln_values, L, rng)
            means[b] = float(np.mean(boot))
        lo, hi = np.quantile(means, [0.025, 0.975])
        candidates.append((float(lo), float(hi)))
    widths = [hi - lo for lo, hi in candidates]
    lo_ln, hi_ln = candidates[int(np.argmax(widths))]
    return float(np.exp(point_ln)), float(np.exp(lo_ln)), float(np.exp(hi_ln))


def mbb_ci_quantile(values: np.ndarray, q: float, draws: int, block_lengths: Iterable[int], seed: int) -> tuple[float, float, float]:
    """q-quantile point and 95% CI via MBB on raw values; choose widest CI."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.nan, np.nan, np.nan
    point = float(np.quantile(values, q))
    rng = np.random.default_rng(seed + 17)
    candidates = []
    for L in block_lengths:
        est = np.empty(draws, dtype=float)
        for b in range(draws):
            boot = _mbb_resample(values, L, rng)
            est[b] = float(np.quantile(boot, q))
        lo, hi = np.quantile(est, [0.025, 0.975])
        candidates.append((float(lo), float(hi)))
    widths = [hi - lo for lo, hi in candidates]
    lo, hi = candidates[int(np.argmax(widths))]
    return point, lo, hi


def analyze_latency_mbb(codec: str, pattern: str, ref_format: str = REF_FORMAT) -> None:
    runs = load_latency_runs(DATA_DIR, codec, pattern)
    if runs.empty:
        raise ValueError(f"No runs for {codec}/{pattern}.")
    token = f"{pattern}_{codec}"

    # Per-format GM (ms) with dependence-robust CI; p95 (ms) with MBB CI
    summary_rows = []
    for fmt, grp in runs.groupby("format"):
        ln_vals = grp["ln_time"].to_numpy()
        lat_ms = 1000.0 * grp["time_sec"].to_numpy()
        gm_s, gm_lo_s, gm_hi_s = mbb_ci_mean_log(ln_vals, draws=B, block_lengths=BLOCK_LENGTHS, seed=BASE_SEED + hash(fmt) % 10_000)
        gm_ms = 1000.0 * gm_s
        gm_lo_ms = 1000.0 * gm_lo_s
        gm_hi_ms = 1000.0 * gm_hi_s
        p95, p95_lo, p95_hi = mbb_ci_quantile(lat_ms, q=0.95, draws=B, block_lengths=BLOCK_LENGTHS, seed=BASE_SEED + 3_000 + hash(fmt) % 10_000)
        summary_rows.append({
            "format": fmt,
            "n": int(len(grp)),
            "gm_ms": gm_ms,
            "gm_ci_lo_ms": gm_lo_ms,
            "gm_ci_hi_ms": gm_hi_ms,
            "p95_ms": p95,
            "p95_ci_lo_ms": p95_lo,
            "p95_ci_hi_ms": p95_hi,
        })

    summary = pd.DataFrame(summary_rows).sort_values("gm_ms", ascending=True).reset_index(drop=True)

    # Ratios vs reference (point only; descriptive)
    if ref_format in summary["format"].values:
        ref_val = float(summary.loc[summary["format"] == ref_format, "gm_ms"].iloc[0])
        ref_used = ref_format
    else:
        idx_min = int(summary["gm_ms"].idxmin())
        ref_val = float(summary.loc[idx_min, "gm_ms"])
        ref_used = str(summary.loc[idx_min, "format"])
    summary["gm_ratio_vs_ref"] = summary["gm_ms"] / ref_val
    summary["ratio_reference"] = ref_used

    # Save table
    summary.rename(columns={
        "format": "Format",
        "n": "n",
        "gm_ms": "GM_latency_ms",
        "gm_ci_lo_ms": "GM_ci_lo_ms",
        "gm_ci_hi_ms": "GM_ci_hi_ms",
        "p95_ms": "p95_ms",
        "p95_ci_lo_ms": "p95_ci_lo_ms",
        "p95_ci_hi_ms": "p95_ci_hi_ms",
        "gm_ratio_vs_ref": f"GM_ratio_vs_{ref_used}",
        "ratio_reference": "ratio_reference",
    }).to_csv(TABLE_DIR / f"tab_mbb_{token}_latency.csv", index=False)

    # Figure 1: GM ± CI with raw points
    order = summary["format"].tolist()
    x = np.arange(len(order))
    gm = summary["gm_ms"].to_numpy(float)
    gm_lo = summary["gm_ci_lo_ms"].to_numpy(float)
    gm_hi = summary["gm_ci_hi_ms"].to_numpy(float)
    err_low = np.maximum(0.0, gm - gm_lo)
    err_high = np.maximum(0.0, gm_hi - gm)
    yerr = np.vstack([err_low, err_high])

    plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
    plt.errorbar(x, gm, yerr=yerr, fmt="o", capsize=5, elinewidth=1.6)
    # raw points overlay (jittered)
    for xi, fmt in zip(x, order):
        pts = 1000.0 * runs.loc[runs["format"] == fmt, "time_sec"].to_numpy(float)
        jitter = np.random.default_rng(BASE_SEED + 42 + xi).uniform(-0.10, 0.10, size=pts.size)
        plt.plot(np.full_like(pts, xi) + jitter, pts, "o", markersize=2, alpha=0.5)
    plt.xticks(x, order)
    plt.ylabel("Geometric mean latency (ms)")
    plt.title(f"{pattern} — {codec}: GM latency ±95% CI (dependence-robust)")
    plt.gca().yaxis.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_gm_ci_ms.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_gm_ci_ms.pdf")
    plt.close()

    # Figure 2: p95 ± CI
    p95 = summary["p95_ms"].to_numpy(float)
    p95_lo = summary["p95_ci_lo_ms"].to_numpy(float)
    p95_hi = summary["p95_ci_hi_ms"].to_numpy(float)
    err95_low = np.maximum(0.0, p95 - p95_lo)
    err95_high = np.maximum(0.0, p95_hi - p95)
    yerr95 = np.vstack([err95_low, err95_high])

    plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
    plt.errorbar(x, p95, yerr=yerr95, fmt="o", capsize=5, elinewidth=1.6)
    plt.xticks(x, order)
    plt.ylabel("p95 latency (ms)")
    plt.title(f"{pattern} — {codec}: p95 latency ±95% CI (block bootstrap)")
    plt.gca().yaxis.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_p95_ci_ms.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_p95_ci_ms.pdf")
    plt.close()

    # Figure 3: ratios vs reference (point only; descriptive)
    ratios = summary["gm_ratio_vs_ref"].to_numpy(float)
    plt.figure(figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
    plt.plot(x, ratios, "o")
    plt.axhline(1.0, linestyle="--", linewidth=1)
    plt.xticks(x, order)
    plt.ylabel(f"GM ratio vs {ref_used} (×)")
    plt.title(f"{pattern} — {codec}: Descriptive GM ratios (session-conditional)")
    plt.gca().yaxis.grid(True, alpha=0.35)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_ratios_point.png", dpi=200)
    plt.savefig(FIGURE_DIR / f"fig_mbb_{token}_ratios_point.pdf")
    plt.close()

In [33]:
# analyze_latency_mbb("zstd", "full")
for codec in CODECS:
    for pattern in PATTERNS:
        analyze_latency_mbb(codec, pattern)

# Latency CloudWatch

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GRAPH_DIR = Path("figures/artifacts")
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

# assumed globals elsewhere:
# CODECS  = ["gzip", "lz4", "zstd"]
# FORMATS = ["hdf5", "zarr", "tiledb", "root"]
# TEXTWIDTH_IN defined elsewhere (≈ \textwidth in inches)
CROSS_HEIGHT_IN = 7.0   # good for 3 stacked rows

def load_mbb_table(codec: str, pattern: str) -> pd.DataFrame:
    """Load dependence-robust GM and p95 table produced by analyze_latency_mbb."""
    token = f"{pattern}_{codec}"
    path = GRAPH_DIR / "tables" / f"tab_mbb_{token}_latency.csv"
    print(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing MBB table: {path}")
    df = pd.read_csv(path)
    # Normalize column names
    cols = {c: c.strip() for c in df.columns}
    df = df.rename(columns=cols)
    if "Format" in df.columns:
        df = df.rename(columns={"Format": "format"})
    return df[[
        "format",
        "GM_latency_ms", "GM_ci_lo_ms", "GM_ci_hi_ms",
        "p95_ms",        "p95_ci_lo_ms", "p95_ci_hi_ms",
    ]].copy()


def build_cross_slice_latency_vs_gets(request_tbl: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for codec in CODECS:
        mbb = load_mbb_table(codec, "slice")
        # request intensity for this codec/pattern
        req = request_tbl[
            (request_tbl["codec"] == codec) &
            (request_tbl["pattern"] == "slice")
        ].copy()
        req = req[["format", "codec", "pattern", "GETs_per_GiB"]].copy()
        merged = mbb.merge(req, on="format", how="inner")
        merged["codec"] = codec
        merged["pattern"] = "slice"
        rows.append(merged)

    frame = pd.concat(rows, ignore_index=True)
    frame["format"] = pd.Categorical(frame["format"],
                                     categories=FORMATS,
                                     ordered=True)
    frame = frame.sort_values(["codec", "format"]).reset_index(drop=True)
    return frame


def build_cross_scan_throughput_vs_gets(request_tbl: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for codec in CODECS:
        mbb = load_mbb_table(codec, "full")

        req = request_tbl[
            (request_tbl["codec"] == codec) &
            (request_tbl["pattern"] == "full")
        ].copy()

        # Per-run downloaded GiB (from CloudWatch aggregate divided by included runs)
        req["GiB_per_rep"] = req["BytesDownloadedSum_GiB"] / req["n_included"]

        merged = mbb.merge(
            req[["format", "codec", "pattern", "GETs_per_GiB", "GiB_per_rep"]],
            on="format", how="inner"
        )

        # Throughput = GiB_per_rep / GM_time_seconds
        merged["GM_time_sec"] = merged["GM_latency_ms"] / 1000.0
        merged["GM_throughput_GiB_s"] = merged["GiB_per_rep"] / merged["GM_time_sec"]

        # CI for throughput via transforming time CI
        merged["GM_throughput_lo"] = merged["GiB_per_rep"] / (merged["GM_ci_hi_ms"] / 1000.0)
        merged["GM_throughput_hi"] = merged["GiB_per_rep"] / (merged["GM_ci_lo_ms"] / 1000.0)

        merged["codec"] = codec
        merged["pattern"] = "full"
        rows.append(merged)

    frame = pd.concat(rows, ignore_index=True)
    frame["format"] = pd.Categorical(frame["format"],
                                     categories=FORMATS,
                                     ordered=True)
    frame = frame.sort_values(["codec", "format"]).reset_index(drop=True)
    return frame


def scatter_grid_slice_latency_vs_gets(frame: pd.DataFrame, out_prefix: str) -> None:
    """3 rows × 1 column: one row per codec, shared x and y."""
    fig, axes = plt.subplots(
        nrows=len(CODECS), ncols=1,
        figsize=(TEXTWIDTH_IN, CROSS_HEIGHT_IN),
        sharex=True, sharey=True,
    )
    if len(CODECS) == 1:
        axes = [axes]

    for ax, codec in zip(axes, CODECS):
        sub = frame[frame["codec"] == codec].copy()
        if sub.empty:
            ax.axis("off")
            continue

        x = sub["GETs_per_GiB"].to_numpy(dtype=float)
        y = sub["GM_latency_ms"].to_numpy(dtype=float)
        y_lo = sub["GM_ci_lo_ms"].to_numpy(dtype=float)
        y_hi = sub["GM_ci_hi_ms"].to_numpy(dtype=float)
        yerr = np.vstack([np.maximum(0, y - y_lo),
                          np.maximum(0, y_hi - y)])

        ax.errorbar(x, y, yerr=yerr, fmt="o", capsize=4, elinewidth=1.2)

        # Compute small offsets in data coordinates to avoid overlap
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        dx = 0.01 * (x_max - x_min)
        dy = 0.02 * (y_max - y_min)

        # annotate points with format name slightly above/right of the marker
        for xi, yi, fmt_name in zip(x, y, sub["format"].astype(str)):
            ax.text(xi + dx, yi + dy, fmt_name,
                    ha="left", va="bottom", fontsize=7)

        ax.set_title(codec)
        ax.yaxis.grid(True, alpha=0.3)

    axes[-1].set_xlabel("GETs per GiB")
    fig.text(0.02, 0.5, "GM latency (ms)", va="center",
             rotation="vertical")
    fig.suptitle("Slice: latency vs GETs per GiB (per session)")

    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf")
    plt.savefig(GRAPH_DIR   / f"{out_prefix}.pdf")
    plt.close(fig)


def scatter_grid_scan_throughput_vs_gets(frame: pd.DataFrame, out_prefix: str) -> None:
    """3 rows × 1 column: one row per codec, shared x and y."""
    fig, axes = plt.subplots(
        nrows=len(CODECS), ncols=1,
        figsize=(TEXTWIDTH_IN, CROSS_HEIGHT_IN),
        sharex=True, sharey=True,
    )
    if len(CODECS) == 1:
        axes = [axes]

    for ax, codec in zip(axes, CODECS):
        sub = frame[frame["codec"] == codec].copy()
        if sub.empty:
            ax.axis("off")
            continue

        x = sub["GETs_per_GiB"].to_numpy(dtype=float)
        y = sub["GM_throughput_GiB_s"].to_numpy(dtype=float)
        y_lo = sub["GM_throughput_lo"].to_numpy(dtype=float)
        y_hi = sub["GM_throughput_hi"].to_numpy(dtype=float)
        yerr = np.vstack([np.maximum(0, y - y_lo),
                          np.maximum(0, y_hi - y)])

        ax.errorbar(x, y, yerr=yerr, fmt="o", capsize=4, elinewidth=1.2)

        # Offsets in data coordinates
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        dx = 0.01 * (x_max - x_min)
        dy = 0.02 * (y_max - y_min)

        for xi, yi, fmt_name in zip(x, y, sub["format"].astype(str)):
            ax.text(xi + dx, yi + dy, fmt_name,
                    ha="left", va="bottom", fontsize=7)

        ax.set_title(codec)
        ax.yaxis.grid(True, alpha=0.3)

    axes[-1].set_xlabel("GETs per GiB")
    fig.text(0.02, 0.5, "GM throughput (GiB/s)", va="center",
             rotation="vertical")
    fig.suptitle("Full scan: throughput vs GETs per GiB (per session)")

    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220)
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf")
    plt.savefig(GRAPH_DIR   / f"{out_prefix}.pdf")
    plt.close(fig)




# Build tables
req_tbl = compute_request_intensity_table(DATA_DIR)

slice_cross = build_cross_slice_latency_vs_gets(req_tbl)
slice_cross.to_csv(TABLE_DIR / "tab_cross_slice_latency_vs_GETsGiB.csv",
                   index=False)

scan_cross = build_cross_scan_throughput_vs_gets(req_tbl)
scan_cross.to_csv(TABLE_DIR / "tab_cross_full_throughput_vs_GETsGiB.csv",
                  index=False)

# Figures
scatter_grid_slice_latency_vs_gets(
    slice_cross,
    out_prefix="fig_cross_slice_latency_vs_GETsGiB_grid",
)
scatter_grid_scan_throughput_vs_gets(
    scan_cross,
    out_prefix="fig_cross_full_throughput_vs_GETsGiB_grid",
)

NameError: name 'compute_request_intensity_table' is not defined

In [12]:
GRAPH_DIR = Path("figures/artifacts")
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

def load_mbb_table(codec: str, pattern: str) -> pd.DataFrame:
    """Load dependence-robust GM and p95 table produced by analyze_latency_mbb."""
    token = f"{pattern}_{codec}"
    path = GRAPH_DIR / "tables" / f"tab_mbb_{token}_latency.csv"
    print(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing MBB table: {path}")
    df = pd.read_csv(path)
    # Normalize column names
    cols = {c: c.strip() for c in df.columns}
    df = df.rename(columns=cols)
    if "Format" in df.columns:
        df = df.rename(columns={"Format": "format"})
    return df[["format", "GM_latency_ms", "GM_ci_lo_ms", "GM_ci_hi_ms",
               "p95_ms", "p95_ci_lo_ms", "p95_ci_hi_ms"]].copy()

def build_cross_slice_latency_vs_gets(request_tbl: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for codec in CODECS:
        try:
            mbb = load_mbb_table(codec, "slice")
        except FileNotFoundError:
            continue
        # request intensity for this codec/pattern
        req = request_tbl[(request_tbl["codec"] == codec) & (request_tbl["pattern"] == "slice")].copy()
        if req.empty:
            continue
        req = req[["format", "codec", "pattern", "GETs_per_GiB"]].copy()
        merged = mbb.merge(req, on="format", how="inner")
        merged["codec"] = codec
        merged["pattern"] = "slice"
        rows.append(merged)
    if not rows:
        raise ValueError("No data for slice cross-metric view.")
    frame = pd.concat(rows, ignore_index=True)
    # keep only requested formats order
    frame["format"] = pd.Categorical(frame["format"], categories=FORMATS, ordered=True)
    frame = frame.sort_values(["codec", "format"]).reset_index(drop=True)
    return frame

def build_cross_scan_throughput_vs_gets(request_tbl: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for codec in CODECS:
        try:
            mbb = load_mbb_table(codec, "full")
        except FileNotFoundError:
            continue
        req = request_tbl[(request_tbl["codec"] == codec) & (request_tbl["pattern"] == "full")].copy()
        if req.empty:
            continue
        # Per-run downloaded GiB (from CloudWatch aggregate divided by included runs)
        req["GiB_per_rep"] = req["BytesDownloadedSum_GiB"] / req["n_included"]
        merged = mbb.merge(req[["format", "codec", "pattern", "GETs_per_GiB", "GiB_per_rep"]], on="format", how="inner")
        # Throughput = GiB_per_rep / GM_time_seconds
        merged["GM_time_sec"] = merged["GM_latency_ms"] / 1000.0
        merged["GM_throughput_GiB_s"] = merged["GiB_per_rep"] / merged["GM_time_sec"]
        # Rough CI for throughput by transforming time CI (optional; shown as interval on y if desired)
        merged["GM_throughput_lo"] = merged["GiB_per_rep"] / (merged["GM_ci_hi_ms"] / 1000.0)
        merged["GM_throughput_hi"] = merged["GiB_per_rep"] / (merged["GM_ci_lo_ms"] / 1000.0)
        merged["codec"] = codec
        merged["pattern"] = "full"
        rows.append(merged)
    if not rows:
        raise ValueError("No data for full-scan cross-metric view.")
    frame = pd.concat(rows, ignore_index=True)
    frame["format"] = pd.Categorical(frame["format"], categories=FORMATS, ordered=True)
    frame = frame.sort_values(["codec", "format"]).reset_index(drop=True)
    return frame

def _compute_global_y_limits_slice(frame: pd.DataFrame, y_pad_frac=0.18):
    y = frame["GM_latency_ms"].to_numpy(float)
    err_hi = np.maximum(0.0, frame["GM_ci_hi_ms"].to_numpy(float) - y)
    ymin = float(np.nanmin(y))
    ymax = float(np.nanmax(y + err_hi))
    span = max(ymax - ymin, 1e-9)
    return ymin - 0.08*span, ymax + y_pad_frac*span

def _compute_global_y_limits_scan(frame: pd.DataFrame, y_pad_frac=0.18):
    y = frame["GM_throughput_GiB_s"].to_numpy(float)
    err_hi = np.maximum(0.0, frame["GM_throughput_hi"].to_numpy(float) - y)
    ymin = float(np.nanmin(y))
    ymax = float(np.nanmax(y + err_hi))
    span = max(ymax - ymin, 1e-9)
    return ymin - 0.08*span, ymax + y_pad_frac*span


def scatter_grid_slice_latency_vs_gets(frame: pd.DataFrame, out_prefix: str) -> None:
    """3 rows × 1 column: one row per codec, shared x and y."""
    fig, axes = plt.subplots(
        nrows=len(CODECS),
        ncols=1,
        figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN),
        sharex=True,
        sharey=True,
    )
    if len(CODECS) == 1:
        axes = [axes]

    ylo_g, yhi_g = _compute_global_y_limits_slice(frame)

    for ax, codec in zip(axes, CODECS):
        sub = frame[frame["codec"] == codec]
        if sub.empty:
            ax.axis("off")
            continue

        x = sub["GETs_per_GiB"].to_numpy(float)
        y = sub["GM_latency_ms"].to_numpy(float)
        y_lo = sub["GM_ci_lo_ms"].to_numpy(float)
        y_hi = sub["GM_ci_hi_ms"].to_numpy(float)

        err_low = np.maximum(0.0, y - y_lo)
        err_high = np.maximum(0.0, y_hi - y)

        ax.errorbar(
            x, y,
            yerr=np.vstack([err_low, err_high]),
            fmt="o",
            capsize=4,
            elinewidth=1.4,
        )
        ax.set_ylim(ylo_g, yhi_g)
        ax.set_title(codec)
        ax.yaxis.grid(True, alpha=0.3)

        pad = 0.02 * (yhi_g - ylo_g)
        for xi, yi, ei, lab in zip(
            x, y, err_high, sub["format"].astype(str)
        ):
            ty = min(max(yi + ei + pad, ylo_g + pad), yhi_g - pad)
            ax.text(
                xi,
                ty,
                lab,
                ha="center",
                va="bottom",
                bbox=dict(
                    facecolor="white",
                    alpha=0.6,
                    edgecolor="none",
                    pad=1.5,
                ),
                clip_on=True,
            )

    # Shared axis labels and title
    axes[-1].set_xlabel("GETs per GiB")
    fig.text(0.02, 0.5, "GM latency (ms)", va="center", rotation="vertical")
    fig.suptitle("Slice: latency vs GETs per GiB (per session)")

    fig.tight_layout(rect=[0.03, 0, 1, 1])

    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.savefig(GRAPH_DIR   / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.close(fig)


def scatter_grid_scan_throughput_vs_gets(frame: pd.DataFrame, out_prefix: str) -> None:
    """3 rows × 1 column: one row per codec, shared x and y."""
    fig, axes = plt.subplots(
        nrows=len(CODECS),
        ncols=1,
        figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN),
        sharex=True,
        sharey=True,
    )
    if len(CODECS) == 1:
        axes = [axes]

    ylo_g, yhi_g = _compute_global_y_limits_scan(frame)

    for ax, codec in zip(axes, CODECS):
        sub = frame[frame["codec"] == codec]
        if sub.empty:
            ax.axis("off")
            continue

        x = sub["GETs_per_GiB"].to_numpy(float)
        y = sub["GM_throughput_GiB_s"].to_numpy(float)
        y_lo = sub["GM_throughput_lo"].to_numpy(float)
        y_hi = sub["GM_throughput_hi"].to_numpy(float)

        err_low = np.maximum(0.0, y - y_lo)
        err_high = np.maximum(0.0, y_hi - y)

        ax.errorbar(
            x, y,
            yerr=np.vstack([err_low, err_high]),
            fmt="o",
            capsize=4,
            elinewidth=1.4,
        )
        ax.set_ylim(ylo_g, yhi_g)
        ax.set_title(codec)
        ax.yaxis.grid(True, alpha=0.3)

        pad = 0.02 * (yhi_g - ylo_g)
        for xi, yi, ei, lab in zip(
            x, y, err_high, sub["format"].astype(str)
        ):
            ty = min(max(yi + ei + pad, ylo_g + pad), yhi_g - pad)
            ax.text(
                xi,
                ty,
                lab,
                ha="center",
                va="bottom",
                bbox=dict(
                    facecolor="white",
                    alpha=0.6,
                    edgecolor="none",
                    pad=1.5,
                ),
                clip_on=True,
            )

    axes[-1].set_xlabel("GETs per GiB")
    fig.text(0.02, 0.5, "GM throughput (GiB/s)", va="center", rotation="vertical")
    fig.suptitle("Full scan: throughput vs GETs per GiB (per session)")
    fig.tight_layout(rect=[0.03, 0, 1, 1])

    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.savefig(GRAPH_DIR   / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.close(fig)






# Build tables
req_tbl = compute_request_intensity_table(DATA_DIR)

slice_cross = build_cross_slice_latency_vs_gets(req_tbl)
slice_cross.to_csv(TABLE_DIR / "tab_cross_slice_latency_vs_GETsGiB.csv", index=False)

scan_cross = build_cross_scan_throughput_vs_gets(req_tbl)
scan_cross.to_csv(TABLE_DIR / "tab_cross_full_throughput_vs_GETsGiB.csv", index=False)

# Figures
scatter_grid_slice_latency_vs_gets(slice_cross, out_prefix="fig_cross_slice_latency_vs_GETsGiB_grid")
scatter_grid_scan_throughput_vs_gets(scan_cross, out_prefix="fig_cross_full_throughput_vs_GETsGiB_grid")

figures/artifacts/tables/tab_mbb_slice_gzip_latency.csv
figures/artifacts/tables/tab_mbb_slice_lz4_latency.csv
figures/artifacts/tables/tab_mbb_slice_zstd_latency.csv
figures/artifacts/tables/tab_mbb_full_gzip_latency.csv
figures/artifacts/tables/tab_mbb_full_lz4_latency.csv
figures/artifacts/tables/tab_mbb_full_zstd_latency.csv


# Verification

In [19]:
from typing import Iterable

NP_REF_FORMAT = "zarr"
NP_BLOCK_LENGTHS = np.array([4, 6, 8, 10], dtype=int)
NP_DRAWS = BOOTSTRAP_DRAWS
NP_BASE = RNG_SEED + 7_000

def _mbb_median_ci_ms(values_ms: np.ndarray,
                      draws: int,
                      block_lengths: Iterable[int],
                      seed: int) -> tuple[float, float, float]:
    """Median (ms) and 95% CI via moving-block bootstrap (no conditionals)."""
    v = np.asarray(values_ms, dtype=float)
    n = v.size
    rng = np.random.default_rng(seed)
    Ls = np.array(list(block_lengths), dtype=int)
    lo = np.empty(Ls.size, dtype=float)
    hi = np.empty(Ls.size, dtype=float)
    for idx, L in enumerate(Ls):
        m = int(np.ceil(n / L))
        starts = rng.integers(0, n, size=(draws, m))
        offsets = np.arange(L, dtype=int)[None, None, :]
        idxs = (starts[:, :, None] + offsets) % n
        boot = v[idxs].reshape(draws, m * L)[:, :n]
        est = np.median(boot, axis=1)
        q = np.quantile(est, [0.025, 0.975])
        lo[idx] = q[0]
        hi[idx] = q[1]
    k = int(np.argmax(hi - lo))
    return float(np.median(v)), float(lo[k]), float(hi[k])

def _load_gm_table(codec: str, pattern: str) -> pd.DataFrame:
    token = f"{pattern}_{codec}"
    path = Path("figures/artifacts/tables")
    df = pd.read_csv(path / f"tab_mbb_{token}_latency.csv")
    cols = {c: c.strip() for c in df.columns}
    df = df.rename(columns=cols)
    df = df.rename(columns={"Format": "format"}) if "Format" in df.columns else df
    return df[["format", "GM_latency_ms"]].copy()

def nonparametric_confirmation_run(codec: str, pattern: str, ref_format: str = NP_REF_FORMAT) -> pd.DataFrame:
    runs = load_latency_runs(DATA_DIR, codec, pattern)
    token = f"{pattern}_{codec}"
    rows = []
    for fmt, grp in runs.groupby("format"):
        lat_ms = 1000.0 * grp["time_sec"].to_numpy(float)
        med, lo, hi = _mbb_median_ci_ms(
            lat_ms,
            draws=NP_DRAWS,
            block_lengths=NP_BLOCK_LENGTHS,
            seed=NP_BASE + 101 * CODECS.index(codec) + 17 * PATTERNS.index(pattern) + FORMATS.index(fmt),
        )
        rows.append({"format": fmt, "n": int(lat_ms.size),
                     "median_ms": med, "median_ci_lo_ms": lo, "median_ci_hi_ms": hi})
    summary = pd.DataFrame(rows).sort_values("median_ms", ascending=True).reset_index(drop=True)

    ref_val = float(summary.set_index("format").loc[ref_format, "median_ms"])
    summary["median_ratio_vs_ref"] = summary["median_ms"] / ref_val
    summary["ratio_reference"] = ref_format

    out_tbl = summary.rename(columns={
        "format": "Format",
        "n": "n",
        "median_ms": "Median_latency_ms",
        "median_ci_lo_ms": "Median_ci_lo_ms",
        "median_ci_hi_ms": "Median_ci_hi_ms",
        "median_ratio_vs_ref": f"Median_ratio_vs_{ref_format}",
        "ratio_reference": "ratio_reference",
    })
    out_tbl.to_csv(TABLE_DIR / f"tab_np_{token}_median.csv", index=False)

    order = summary["format"].tolist()
    x = np.arange(len(order))
    med = summary["median_ms"].to_numpy(float)
    med_lo = summary["median_ci_lo_ms"].to_numpy(float)
    med_hi = summary["median_ci_hi_ms"].to_numpy(float)
    err_low = np.maximum(0.0, med - med_lo)
    err_high = np.maximum(0.0, med_hi - med)
    yerr = np.vstack([err_low, err_high])

    # plt.figure(figsize=(6.6, 3.8))
    # plt.errorbar(x, med, yerr=yerr, fmt="o", capsize=5, elinewidth=1.6)
    # for xi, fmt in zip(x, order):
    #     pts = 1000.0 * runs.loc[runs["format"] == fmt, "time_sec"].to_numpy(float)
    #     jitter = np.random.default_rng(NP_BASE + 42 + xi).uniform(-0.10, 0.10, size=pts.size)
    #     plt.plot(np.full_like(pts, xi) + jitter, pts, "o", markersize=2, alpha=0.5)
    # plt.xticks(x, order)
    # plt.ylabel("Median latency (ms)")
    # plt.title(f"{pattern} — {codec}: Median latency ±95% CI (MBB)")
    # plt.gca().yaxis.grid(True, alpha=0.35)
    # plt.tight_layout()
    # plt.savefig(FIGURE_DIR / f"fig_np_{token}_median_ci_ms.png", dpi=220)
    # plt.savefig(FIGURE_DIR / f"fig_np_{token}_median_ci_ms.pdf")
    # plt.close()

    ratios = summary["median_ratio_vs_ref"].to_numpy(float)
    # plt.figure(figsize=(6.0, 3.2))
    # plt.plot(x, ratios, "o")
    # plt.axhline(1.0, linestyle="--", linewidth=1)
    # plt.xticks(x, order)
    # plt.ylabel(f"Median ratio vs {ref_format} (×)")
    # plt.title(f"{pattern} — {codec}: Descriptive median ratios (session-conditional)")
    # plt.gca().yaxis.grid(True, alpha=0.35)
    # plt.tight_layout()
    # plt.savefig(FIGURE_DIR / f"fig_np_{token}_ratios_point.png", dpi=200)
    # plt.savefig(FIGURE_DIR / f"fig_np_{token}_ratios_point.pdf")
    # plt.close()

    gm_tbl = _load_gm_table(codec, pattern)
    med_rank = summary.set_index("format")["median_ms"].rank(method="min", ascending=True)
    gm_rank = gm_tbl.set_index("format")["GM_latency_ms"].rank(method="min", ascending=True)
    comp = pd.concat([med_rank.rename("rank_median"), gm_rank.rename("rank_gm")], axis=1)
    comp["rank_diff"] = comp["rank_median"] - comp["rank_gm"]
    comp.reset_index().rename(columns={"index": "Format"}).to_csv(
        TABLE_DIR / f"tab_np_{token}_rank_compare.csv", index=False
    )

    return summary

def nonparametric_ratios_grid(pattern: str, out_prefix: str, ref_format: str = NP_REF_FORMAT) -> None:
    # 3 rows × 1 column: one row per codec, shared y
    fig, axes = plt.subplots(
        nrows=len(CODECS),
        ncols=1,
        figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN),
        sharex=False,
        sharey=True,
    )
    axes = np.atleast_1d(axes)

    for ax, codec in zip(axes, CODECS):
        path = TABLE_DIR / f"tab_np_{pattern}_{codec}_median.csv"
        if not path.exists():
            ax.axis("off")
            continue

        tbl = pd.read_csv(path)
        cols = {c: c.strip() for c in tbl.columns}
        tbl = tbl.rename(columns=cols)
        if "Format" in tbl.columns:
            tbl = tbl.rename(columns={"Format": "format"})

        ratio_col = [c for c in tbl.columns
                     if c.lower().startswith("median_ratio_vs_")][0]

        x = np.arange(len(tbl))
        y = tbl[ratio_col].to_numpy(float)

        ax.plot(x, y, "o")
        ax.axhline(1.0, linestyle="--", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(tbl["format"])
        ax.set_title(codec)
        ax.yaxis.grid(True, alpha=0.35)

    axes[1].set_ylabel(f"Median ratio vs {ref_format} (×)")
    axes[-1].set_xlabel("format")
    fig.suptitle(f"{pattern}: descriptive median ratios by codec")

    fig.tight_layout(rect=[0.03, 0.0, 1.0, 1.0])

    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.savefig(GRAPH_DIR   / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.close(fig)

# %% Run for all codec×pattern and build grids (no conditionals)
for pattern in PATTERNS:
    for codec in CODECS:
        nonparametric_confirmation_run(codec, pattern, ref_format=NP_REF_FORMAT)
    nonparametric_ratios_grid(pattern, out_prefix=f"fig_np_{pattern}_ratios_grid", ref_format=NP_REF_FORMAT)


In [23]:
# %% [markdown]
# # Appendix: Nonparametric confirmation — per-pattern grids (1×3)

# %%
from pathlib import Path

NP_REF_FORMAT = "zarr"  # keep consistent

# Ensure graphs directory for LaTeX paths
GRAPHS_DIR = Path("figures/graphs")
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

def _np_load_tbl(codec: str, pattern: str) -> pd.DataFrame:
    token = f"{pattern}_{codec}"
    df = pd.read_csv(TABLE_DIR / f"tab_np_{token}_median.csv")
    df = df.rename(columns={c: c.strip() for c in df.columns})
    df = df.rename(columns={"Format": "format"})
    return df

def _clamped_stacked_positions(y_base: np.ndarray,
                               y_top: np.ndarray,
                               ylo: float, yhi: float,
                               pad_frac: float = 0.04,
                               min_sep_frac: float = 0.06) -> np.ndarray:
    span = max(yhi - ylo, 1e-9)
    margin = pad_frac * span
    min_sep = min_sep_frac * span
    pos = np.maximum(y_top, y_base + margin)
    order = np.argsort(pos)
    for k in range(1, len(order)):
        prev, cur = order[k - 1], order[k]
        if pos[cur] - pos[prev] < min_sep:
            pos[cur] = pos[prev] + min_sep
    overflow = float(np.max(pos) - (yhi - margin))
    if overflow > 0:
        pos -= overflow
    pos = np.clip(pos, ylo + margin, yhi - margin)
    return pos

def _clamp_x_with_align(x: np.ndarray, xmin: float, xmax: float, pad_frac: float = 0.06):
    span = xmax - xmin
    pad = pad_frac * span
    xp = np.clip(x, xmin + pad, xmax - pad)
    ha = []
    for xi in xp:
        if xi <= xmin + 2*pad:
            ha.append("left")
        elif xi >= xmax - 2*pad:
            ha.append("right")
        else:
            ha.append("center")
    return xp, ha

def _pattern_y_limits(pattern: str) -> tuple[float, float]:
    ymins, ymaxs = [], []
    for codec in CODECS:
        df = _np_load_tbl(codec, pattern)
        ymins.append(np.min(df["Median_latency_ms"]))
        ymaxs.append(np.max(df["Median_ci_hi_ms"]))
    ylo = float(np.min(ymins)); yhi = float(np.max(ymaxs))
    span = max(yhi - ylo, 1e-9)
    # generous padding to keep labels off borders
    return ylo - 0.12*span, yhi + 0.26*span

def np_median_ci_row_grid(pattern: str, out_prefix: str) -> None:
    """Three rows × one column (codecs) for a single pattern: median ± CI with safe labels."""
    ylo, yhi = _pattern_y_limits(pattern)

    fig, axes = plt.subplots(
        nrows=len(CODECS),
        ncols=1,
        figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN),
        sharey=True,
    )
    axes = np.atleast_1d(axes)

    for idx, (ax, codec) in enumerate(zip(axes, CODECS)):
        df = _np_load_tbl(codec, pattern).set_index("format").reindex(FORMATS).reset_index()

        x = np.arange(len(df))
        y = df["Median_latency_ms"].to_numpy(float)
        lo = df["Median_ci_lo_ms"].to_numpy(float)
        hi = df["Median_ci_hi_ms"].to_numpy(float)
        err_low = np.maximum(0.0, y - lo)
        err_high = np.maximum(0.0, hi - y)

        # widen x-limits to avoid edge collisions
        xpad = 0.35
        xmin, xmax = -0.5 - xpad, (len(df) - 0.5) + xpad
        ax.set_xlim(xmin, xmax)

        ax.errorbar(
            x, y,
            yerr=np.vstack([err_low, err_high]),
            fmt="o",
            capsize=5,
            elinewidth=1.6,
        )
        ax.set_ylim(ylo, yhi)
        ax.set_title(codec)
        ax.yaxis.grid(True, alpha=0.35)

        # x-ticks / labels only on the last row
        if idx == len(CODECS) - 1:
            ax.set_xticks(x)
            ax.set_xticklabels(df["format"])
            ax.set_xlabel("format")
        else:
            ax.set_xticks(x)
            ax.set_xticklabels([])

        # y-label only on the middle row (axes[1])
        if idx == 1:
            ax.set_ylabel(f"{pattern} — median latency (ms)")

        # label positions: above error bars, separated, clamped
        desired_top = y + err_high + 0.02 * (yhi - ylo)
        text_y = _clamped_stacked_positions(
            y,
            desired_top,
            ylo,
            yhi,
            pad_frac=0.04,
            min_sep_frac=0.06,
        )

        raw_x = x + np.linspace(-0.10, 0.10, num=len(x))
        text_x, ha_list = _clamp_x_with_align(
            raw_x,
            xmin,
            xmax,
            pad_frac=0.06,
        )

        for xi, yi, ty, lab, ha in zip(
            text_x, y, text_y, df["format"].astype(str), ha_list
        ):
            ax.text(
                xi,
                ty,
                f"{yi:.0f}",
                ha=ha,
                va="bottom",
                clip_on=True,
                bbox=dict(
                    facecolor="white",
                    edgecolor="none",
                    alpha=0.65,
                    pad=1.6,
                ),
                zorder=5,
            )

    fig.suptitle(f"Median latency ±95% CI (MBB) — {pattern}")
    fig.tight_layout(rect=[0.03, 0.0, 1.0, 1.0])

    # Save to both artifacts and LaTeX graphs path
    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.savefig(GRAPHS_DIR   / f"{out_prefix}.pdf", bbox_inches="tight")
    plt.close(fig)

# Build per-pattern rows
np_median_ci_row_grid(pattern="slice", out_prefix="fig_np_median_ci_row_slice")
np_median_ci_row_grid(pattern="full",  out_prefix="fig_np_median_ci_row_full")

# V2

In [25]:
# %% [markdown]
# # Tail sensitivity (p95 only): HD estimator + MBB CIs, ℓ-grid; per-pattern 1×3 grids with legends

# %%
from scipy.stats.mstats import hdquantiles  # Harrell–Davis estimator

P95 = 0.95
TAIL_LS = np.array([4, 6, 8, 10], dtype=int)
TAIL_DRAWS = BOOTSTRAP_DRAWS
TAIL_SEED_BASE = RNG_SEED + 21_000

def _hd_q(values: np.ndarray, q: float) -> float:
    return float(hdquantiles(values, prob=[q])[0])

def _mbb_hd_ci_byL(values_ms: np.ndarray,
                   q: float,
                   draws: int,
                   Ls: np.ndarray,
                   seed: int) -> pd.DataFrame:
    """Per-L HD point and 95% CI via moving-block bootstrap."""
    v = np.asarray(values_ms, float)
    rng = np.random.default_rng(seed)
    n = v.size
    rows = []
    for L in Ls:
        m = int(np.ceil(n / L))
        starts = rng.integers(0, n, size=(draws, m))
        offsets = np.arange(L, dtype=int)[None, None, :]
        idxs = (starts[:, :, None] + offsets) % n
        boot = v[idxs].reshape(draws, m * L)[:, :n]
        est = np.apply_along_axis(_hd_q, 1, boot, q=q)
        lo, hi = np.quantile(est, [0.025, 0.975])
        rows.append({
            "L": int(L),
            "point_ms": _hd_q(v, q),
            "lo_ms": float(lo),
            "hi_ms": float(hi),
            "width_ms": float(hi - lo),
        })
    return pd.DataFrame(rows)

def _compute_y_limits(df: pd.DataFrame, lo_col: str, hi_col: str, pad_lo=0.12, pad_hi=0.26) -> tuple[float, float]:
    ylo = float(np.nanmin(df[lo_col]))
    yhi = float(np.nanmax(df[hi_col]))
    span = max(yhi - ylo, 1e-9)
    return ylo - pad_lo * span, yhi + pad_hi * span

def tail_sensitivity_p95_tables(codec: str, pattern: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Tables:
      - byL: tab_tail_<pattern>_<codec>_p95_byL.csv (per format, per L)
      - selected: tab_tail_<pattern>_<codec>_p95_selected.csv (widest CI per format)
    """
    runs = load_latency_runs(DATA_DIR, codec, pattern)
    token = f"{pattern}_{codec}"
    base_seed = TAIL_SEED_BASE + 31 * CODECS.index(codec) + 11 * PATTERNS.index(pattern)
    rows_byL = []
    rows_sel = []
    for fmt, grp in runs.groupby("format"):
        lat_ms = 1000.0 * grp["time_sec"].to_numpy(float)
        tbl = _mbb_hd_ci_byL(lat_ms, q=P95, draws=TAIL_DRAWS, Ls=TAIL_LS,
                             seed=base_seed + 1000 * FORMATS.index(fmt))
        tbl["format"] = fmt
        rows_byL.append(tbl)
        k = int(tbl["width_ms"].idxmax())
        sel = tbl.loc[k]
        rows_sel.append({
            "format": fmt,
            "point_ms": float(sel["point_ms"]),
            "ci_lo_ms": float(sel["lo_ms"]),
            "ci_hi_ms": float(sel["hi_ms"]),
            "ci_width_ms": float(sel["width_ms"]),
            "L_selected": int(sel["L"]),
        })
    byL = pd.DataFrame(rows_byL[0]).iloc[0:0].copy() if not rows_byL else pd.concat(rows_byL, ignore_index=True)
    selected = pd.DataFrame(rows_sel)
    byL = byL[["format", "L", "point_ms", "lo_ms", "hi_ms", "width_ms"]].sort_values(["L","format"])
    selected = selected[["format","point_ms","ci_lo_ms","ci_hi_ms","ci_width_ms","L_selected"]].sort_values("format")

    byL.to_csv(TABLE_DIR / f"tab_tail_{token}_p95_byL.csv", index=False)
    selected.to_csv(TABLE_DIR / f"tab_tail_{token}_p95_selected.csv", index=False)
    return byL, selected

def plot_tail_p95_grid_for_pattern(pattern: str, out_prefix: str) -> None:
    """
    Three rows × one column (codecs). y = p95 (HD) with 95% MBB CIs vs block length L.
    Legend (formats) in the top-left of each panel. No per-point text.
    """
    # Colors mapped to formats for consistency across panels
    color_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["C0", "C1", "C2", "C3", "C4"])
    fmt_colors = {fmt: color_cycle[i % len(color_cycle)] for i, fmt in enumerate(FORMATS)}

    fig, axes = plt.subplots(
        nrows=len(CODECS),
        ncols=1,
        figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN),
        sharex=True,
        sharey=False,
    )
    axes = np.atleast_1d(axes)

    for idx, (ax, codec) in enumerate(zip(axes, CODECS)):
        byL, _ = tail_sensitivity_p95_tables(codec, pattern)  # build tables and get data
        if byL.empty:
            ax.axis("off")
            continue

        # y-limits with padding per panel
        ylo, yhi = _compute_y_limits(byL, "lo_ms", "hi_ms")

        # x axis: discrete L with small margin
        xcats = TAIL_LS.astype(int).tolist()
        xpad = 0.25
        ax.set_xlim(min(xcats) - xpad, max(xcats) + xpad)
        ax.set_ylim(ylo, yhi)

        handles = []
        labels = []

        # small horizontal offsets per format to avoid overlap of error bars
        offsets = np.linspace(-0.08, 0.08, num=len(FORMATS))
        for off, fmt in zip(offsets, FORMATS):
            s = byL[byL["format"] == fmt].sort_values("L")
            if s.empty:
                continue
            xs = s["L"].to_numpy(float) + off
            ys = s["point_ms"].to_numpy(float)
            lo = s["lo_ms"].to_numpy(float)
            hi = s["hi_ms"].to_numpy(float)
            err = np.vstack([
                np.maximum(0.0, ys - lo),
                np.maximum(0.0, hi - ys),
            ])
            h = ax.errorbar(
                xs, ys,
                yerr=err,
                fmt="o",
                capsize=4,
                elinewidth=1.2,
                color=fmt_colors[fmt],
                markersize=4,
                linewidth=1.2,
            )
            handles.append(h.lines[0])
            labels.append(fmt)

        ax.set_xticks(TAIL_LS)

        # x tick labels and xlabel only on the last row
        if idx == len(CODECS) - 1:
            ax.set_xticklabels(TAIL_LS.astype(int).tolist())
            ax.set_xlabel(r"Block length $\ell$")
        else:
            ax.set_xticklabels([])

        ax.set_title(codec)
        ax.yaxis.grid(True, alpha=0.35)
        ax.legend(handles, labels, loc="upper left", frameon=True, framealpha=0.85)

        # y-label only on the middle row (if it exists)
        if idx == 1:
            ax.set_ylabel(f"{pattern} — p95 latency (ms)")

    fig.suptitle(f"Tail sensitivity (HD p95) — {pattern}: effect of block length")
    fig.tight_layout(rect=[0.03, 0.0, 1.0, 1.0])

    plt.savefig(FIGURE_DIR / f"{out_prefix}.png", dpi=220, bbox_inches="tight")
    plt.savefig(FIGURE_DIR / f"{out_prefix}.pdf", bbox_inches="tight")

    graphs_dir = Path("figures/graphs")
    graphs_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(graphs_dir / f"{out_prefix}.pdf", bbox_inches="tight")

    plt.close(fig)


# Build per-pattern grids (slice and full)
plot_tail_p95_grid_for_pattern("slice", out_prefix="fig_tail_slice_p95_hd_ci_vs_L")
plot_tail_p95_grid_for_pattern("full",  out_prefix="fig_tail_full_p95_hd_ci_vs_L")

In [26]:
from pathlib import Path

# Inputs produced earlier:
# - Slice GM & p95 (MBB): TABLE_DIR / f"tab_mbb_slice_{codec}_latency.csv"
# - Slice GETs/GiB:       TABLE_DIR / "tab_cross_slice_latency_vs_GETsGiB.csv"
# - Full GETs/GiB+thr:    TABLE_DIR / "tab_cross_full_throughput_vs_GETsGiB.csv"
# - Full GM time (MBB):   TABLE_DIR / f"tab_mbb_full_{codec}_latency.csv"

# Static stored GiB (from compression table)
STORED_GIB = {
    "gzip":  {"hdf5": 11.11, "zarr": 11.09, "tiledb": 11.44, "root": 11.09},
    "lz4":   {"hdf5": 19.17, "zarr": 10.40, "tiledb": 15.56, "root": 13.31},
    "zstd":  {"hdf5": 10.67, "zarr": 10.51, "tiledb": 10.74, "root": 10.26},
}

def _fmt_int(x: float) -> str:
    return f"{x:,.0f}".replace(",", r"\,")

def _fmt_ms_ci(v: float, lo: float, hi: float) -> str:
    return f"{_fmt_int(v)} \\ ([{_fmt_int(lo)}, {_fmt_int(hi)}])"

def _fmt_thr_ci(v: float, lo: float, hi: float) -> str:
    return f"{v:.3f} \\ ([{lo:.3f}, {hi:.3f}])"

def _fmt_float(x: float, nd=1) -> str:
    return f"{x:.{nd}f}"

def _league_slice(codec: str, req_slice: pd.DataFrame) -> pd.DataFrame:
    path = Path("figures/artifacts/tables")
    mbb = pd.read_csv(path / f"tab_mbb_slice_{codec}_latency.csv")
    mbb = mbb.rename(columns={c: c.strip() for c in mbb.columns})
    mbb = mbb.rename(columns={"Format": "format"})
    want = ["format","GM_latency_ms","GM_ci_lo_ms","GM_ci_hi_ms","p95_ms","p95_ci_lo_ms","p95_ci_hi_ms"]
    mbb = mbb[want].copy()

    req = req_slice[req_slice["codec"] == codec].copy()
    req = req[["format","GETs_per_GiB"]]

    stored = pd.DataFrame([{"format": f, "stored_gib": STORED_GIB[codec][f]} for f in FORMATS])

    df = mbb.merge(req, on="format").merge(stored, on="format")
    # ranking: lower GM latency, tie p95, then GETs/GiB, then stored GiB
    df = df.sort_values(
        by=["GM_latency_ms","p95_ms","GETs_per_GiB","stored_gib"],
        ascending=[True, True, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    df.insert(0, "Rank", np.arange(1, len(df)+1))

    # formatted columns
    df["GM (ms) [CI]"]  = [_fmt_ms_ci(v, lo, hi) for v,lo,hi in zip(df.GM_latency_ms, df.GM_ci_lo_ms, df.GM_ci_hi_ms)]
    df["p95 (ms) [CI]"] = [_fmt_ms_ci(v, lo, hi) for v,lo,hi in zip(df.p95_ms, df.p95_ci_lo_ms, df.p95_ci_hi_ms)]
    df["GETs/GiB"]      = df["GETs_per_GiB"].map(lambda x: _fmt_float(x, 1))
    df["Stored GiB"]    = df["stored_gib"].map(lambda x: _fmt_float(x, 2))

    return df[["Rank","format","GM (ms) [CI]","p95 (ms) [CI]","GETs/GiB","Stored GiB"]].rename(columns={"format":"Format"})

def _league_full(codec: str, req_full: pd.DataFrame) -> pd.DataFrame:
    path = Path("figures/artifacts/tables")
    mbb = pd.read_csv(path / f"tab_mbb_full_{codec}_latency.csv")
    mbb = mbb.rename(columns={c: c.strip() for c in mbb.columns})
    mbb = mbb.rename(columns={"Format": "format"})
    mbb = mbb[["format","GM_latency_ms","GM_ci_lo_ms","GM_ci_hi_ms"]]

    req = req_full[req_full["codec"] == codec].copy()
    want = ["format","GETs_per_GiB","GM_throughput_GiB_s","GM_throughput_lo","GM_throughput_hi","GM_time_sec"]
    req = req[want]

    stored = pd.DataFrame([{"format": f, "stored_gib": STORED_GIB[codec][f]} for f in FORMATS])

    df = req.merge(mbb, on="format").merge(stored, on="format")
    # ranking: higher throughput, tie GM time lower, then GETs/GiB lower, then stored GiB lower
    df = df.sort_values(
        by=["GM_throughput_GiB_s","GM_time_sec","GETs_per_GiB","stored_gib"],
        ascending=[False, True, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    df.insert(0, "Rank", np.arange(1, len(df)+1))

    df["GM thr (GiB/s) [CI]"] = [
        _fmt_thr_ci(v, lo, hi) for v,lo,hi in zip(df.GM_throughput_GiB_s, df.GM_throughput_lo, df.GM_throughput_hi)
    ]
    df["GM time (ms) [CI]"] = [
        _fmt_ms_ci(1000*dfv, 1000*lo, 1000*hi)
        for dfv,lo,hi in zip(df.GM_time_sec, df.GM_ci_lo_ms/1000.0, df.GM_ci_hi_ms/1000.0)
    ]
    df["GETs/GiB"]   = df["GETs_per_GiB"].map(lambda x: _fmt_float(x, 1))
    df["Stored GiB"] = df["stored_gib"].map(lambda x: _fmt_float(x, 2))

    return df[["Rank","format","GM thr (GiB/s) [CI]","GM time (ms) [CI]","GETs/GiB","Stored GiB"]].rename(columns={"format":"Format"})

def _to_latex_combined(codec: str, slice_tbl: pd.DataFrame, full_tbl: pd.DataFrame) -> str:
    # Two panels inside one table float
    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{League tables — {codec} (slice = latency, full = throughput).}}")
    lines.append(rf"\label{{tab:league-" + codec + r"}}")

    # Slice panel
    lines.append(r"\textbf{Slice (lower is better)}")
    lines.append(slice_tbl.to_latex(index=False, escape=False))
    lines.append(r"\vspace{0.5em}")

    # Full panel
    lines.append(r"\textbf{Full scan (higher is better)}")
    lines.append(full_tbl.to_latex(index=False, escape=False))

    lines.append(r"\end{table}")
    return "\n".join(lines)

def build_league_tables():
    # load request-intensity cross tables
    path = Path("figures/artifacts/tables")
    slice_req = pd.read_csv(path / "tab_cross_slice_latency_vs_GETsGiB.csv")
    full_req  = pd.read_csv(path / "tab_cross_full_throughput_vs_GETsGiB.csv")

    Path(TABLE_DIR).mkdir(parents=True, exist_ok=True)
    combined_tex_paths = []
    for codec in CODECS:
        slice_tbl = _league_slice(codec, slice_req)
        full_tbl  = _league_full(codec, full_req)

        # Save CSVs too
        slice_tbl.to_csv(TABLE_DIR / f"tab_league_{codec}_slice.csv", index=False)
        full_tbl.to_csv(TABLE_DIR / f"tab_league_{codec}_full.csv", index=False)

        # Save combined LaTeX
        tex = _to_latex_combined(codec, slice_tbl, full_tbl)
        out_path = TABLE_DIR / f"tab_league_{codec}.tex"
        out_path.write_text(tex, encoding="utf-8")
        combined_tex_paths.append(out_path)

    print("Wrote:", *[str(p) for p in combined_tex_paths], sep="\n  ")

# Run
build_league_tables()


Wrote:
  artifacts/tables/tab_league_gzip.tex
  artifacts/tables/tab_league_lz4.tex
  artifacts/tables/tab_league_zstd.tex


# V4

In [ ]:
# Robust min–max within codec×pattern uses 5th–95th percentiles.
SLICE_WEIGHTS = {"gm": 0.60, "p95": 0.20, "gets": 0.10, "stored": 0.10}
FULL_WEIGHTS  = {"thr": 0.60, "gets": 0.15, "stored": 0.15, "p95": 0.10}

# Stored GiB per codec×format (metadata-inclusive), from compression tables
STORED_GIB = {
    "gzip":  {"hdf5": 11.11, "zarr": 11.09, "tiledb": 11.44, "root": 11.09},
    "lz4":   {"hdf5": 19.17, "zarr": 10.40, "tiledb": 15.56, "root": 13.31},
    "zstd":  {"hdf5": 10.67, "zarr": 10.51, "tiledb": 10.74, "root": 10.26},
}

def _robust_bounds(x: pd.Series) -> tuple[float, float]:
    lo = float(np.nanpercentile(x.to_numpy(float), 5))
    hi = float(np.nanpercentile(x.to_numpy(float), 95))
    if not np.isfinite(hi - lo) or abs(hi - lo) < 1e-12:
        # degenerate range; expand minimally
        lo = float(np.nanmin(x.to_numpy(float)))
        hi = float(np.nanmax(x.to_numpy(float))) + 1e-9
    return lo, hi

def _normalize_minimize(x: pd.Series) -> pd.Series:
    lo, hi = _robust_bounds(x)
    s = 100.0 * (hi - x) / (hi - lo)
    return s.clip(0.0, 100.0)

def _normalize_maximize(x: pd.Series) -> pd.Series:
    lo, hi = _robust_bounds(x)
    s = 100.0 * (x - lo) / (hi - lo)
    return s.clip(0.0, 100.0)

def _dominance_flags(score_df: pd.DataFrame, comp_cols: list[str]) -> pd.DataFrame:
    # Determine Pareto dominance within codec×pattern
    fmts = score_df["Format"].tolist()
    dominates = {f: [] for f in fmts}
    dominated_by = {f: [] for f in fmts}
    A = score_df.set_index("Format")[comp_cols].to_numpy(float)
    for i, fi in enumerate(fmts):
        for j, fj in enumerate(fmts):
            if i == j: continue
            # fi dominates fj if fi >= fj on all metrics and > on at least one
            ge_all = np.all(A[i, :] >= A[j, :])
            gt_any = np.any(A[i, :] >  A[j, :])
            if ge_all and gt_any:
                dominates[fi].append(fj)
                dominated_by[fj].append(fi)
    score_df["Dominates"] = score_df["Format"].map(lambda f: ", ".join(dominates[f]) if dominates[f] else "")
    score_df["Dominated by"] = score_df["Format"].map(lambda f: ", ".join(dominated_by[f]) if dominated_by[f] else "")
    return score_df

def _fmt_heatmap(ax, data: np.ndarray, row_labels: list[str], col_labels: list[str],
                 vmin: float = 0.0, vmax: float = 100.0) -> None:
    im = ax.imshow(data, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=0)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    # annotate
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            ax.text(j, i, f"{val:.0f}", ha="center", va="center")

def build_scores_slice(codec: str, req_slice: pd.DataFrame) -> pd.DataFrame:
    path = Path("figures/artifacts/tables")
    mbb = pd.read_csv(path / f"tab_mbb_slice_{codec}_latency.csv")
    mbb = mbb.rename(columns={c: c.strip() for c in mbb.columns})
    if "Format" in mbb.columns:
        mbb = mbb.rename(columns={"Format": "format"})
    mbb = mbb[["format","GM_latency_ms","p95_ms"]]

    req = req_slice[req_slice["codec"] == codec][["format","GETs_per_GiB"]].copy()
    stored = pd.DataFrame([{"format": f, "stored_gib": STORED_GIB[codec][f]} for f in FORMATS])

    df = mbb.merge(req, on="format").merge(stored, on="format")
    df["Score_GM"]     = _normalize_minimize(df["GM_latency_ms"])
    df["Score_p95"]    = _normalize_minimize(df["p95_ms"])
    df["Score_GETs"]   = _normalize_minimize(df["GETs_per_GiB"])
    df["Score_Stored"] = _normalize_minimize(df["stored_gib"])
    df["Score_Total"]  = (
        SLICE_WEIGHTS["gm"]    * df["Score_GM"] +
        SLICE_WEIGHTS["p95"]   * df["Score_p95"] +
        SLICE_WEIGHTS["gets"]  * df["Score_GETs"] +
        SLICE_WEIGHTS["stored"]* df["Score_Stored"]
    )
    out = df[["format","Score_GM","Score_p95","Score_GETs","Score_Stored","Score_Total"]].copy()
    out = out.sort_values("Score_Total", ascending=False).reset_index(drop=True)
    out = out.rename(columns={"format":"Format"})
    out = _dominance_flags(out, ["Score_GM","Score_p95","Score_GETs","Score_Stored"])
    return out

def build_scores_full(codec: str, req_full: pd.DataFrame) -> pd.DataFrame:
    # throughput + GETs/GiB + GM_time_sec from req_full; p95_ms from mbb full
    req = req_full[req_full["codec"] == codec].copy()
    req = req.rename(columns={c: c.strip() for c in req.columns})
    req = req[["format","GETs_per_GiB","GM_throughput_GiB_s"]].copy()

    path = Path("figures/artifacts/tables")
    mbb = pd.read_csv(path / f"tab_mbb_full_{codec}_latency.csv")
    mbb = mbb.rename(columns={c: c.strip() for c in mbb.columns})
    if "Format" in mbb.columns: mbb = mbb.rename(columns={"Format":"format"})
    mbb = mbb[["format","p95_ms","GM_latency_ms"]]

    stored = pd.DataFrame([{"format": f, "stored_gib": STORED_GIB[codec][f]} for f in FORMATS])

    df = req.merge(mbb, on="format").merge(stored, on="format")
    df["Score_Thr"]    = _normalize_maximize(df["GM_throughput_GiB_s"])
    df["Score_GETs"]   = _normalize_minimize(df["GETs_per_GiB"])
    df["Score_Stored"] = _normalize_minimize(df["stored_gib"])
    df["Score_p95"]    = _normalize_minimize(df["p95_ms"])
    df["Score_Total"]  = (
        FULL_WEIGHTS["thr"]    * df["Score_Thr"] +
        FULL_WEIGHTS["gets"]   * df["Score_GETs"] +
        FULL_WEIGHTS["stored"] * df["Score_Stored"] +
        FULL_WEIGHTS["p95"]    * df["Score_p95"]
    )
    out = df[["format","Score_Thr","Score_p95","Score_GETs","Score_Stored","Score_Total"]].copy()
    out = out.sort_values("Score_Total", ascending=False).reset_index(drop=True)
    out = out.rename(columns={"format":"Format"})
    out = _dominance_flags(out, ["Score_Thr","Score_p95","Score_GETs","Score_Stored"])
    return out

def save_scores_and_heatmaps():
    path = Path("figures/artifacts/tables")
    slice_req = pd.read_csv(path / "tab_cross_slice_latency_vs_GETsGiB.csv")
    full_req  = pd.read_csv(path / "tab_cross_full_throughput_vs_GETsGiB.csv")

    for codec in CODECS:
        # Slice
        sc_slice = build_scores_slice(codec, slice_req)
        sc_slice.to_csv(TABLE_DIR / f"tab_scores_slice_{codec}.csv", index=False)

        fig, ax = plt.subplots(1, 1, figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
        data = sc_slice[["Score_GM","Score_p95","Score_GETs","Score_Stored","Score_Total"]].to_numpy(float)
        rows = sc_slice["Format"].tolist()
        cols = ["GM","p95","GETs/GiB","Stored","Total"]
        _fmt_heatmap(ax, data, rows, cols, vmin=0.0, vmax=100.0)
        ax.set_title(f"Scores (Slice) — {codec}")
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"fig_scores_slice_{codec}_heatmap.png", dpi=220)
        plt.savefig(FIGURE_DIR / f"fig_scores_slice_{codec}_heatmap.pdf")
        plt.close(fig)

        # Full
        sc_full = build_scores_full(codec, full_req)
        sc_full.to_csv(TABLE_DIR / f"tab_scores_full_{codec}.csv", index=False)

        fig, ax = plt.subplots(1, 1, figsize=(TEXTWIDTH_IN, QQ_GRID_HEIGHT_IN / 2))
        data = sc_full[["Score_Thr","Score_p95","Score_GETs","Score_Stored","Score_Total"]].to_numpy(float)
        rows = sc_full["Format"].tolist()
        cols = ["Thr","p95","GETs/GiB","Stored","Total"]
        _fmt_heatmap(ax, data, rows, cols, vmin=0.0, vmax=100.0)
        ax.set_title(f"Scores (Full) — {codec}")
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"fig_scores_full_{codec}_heatmap.png", dpi=220)
        plt.savefig(FIGURE_DIR / f"fig_scores_full_{codec}_heatmap.pdf")
        plt.close(fig)

# Execute
save_scores_and_heatmaps()
